<hr>

# 🧫 DATA CLEANING - Valeurs Foncieres


<style>
h1 {
    text-align: left;
    color: blue;
    font-weight: bold;
}

</style>
<hr>

```text
STEPS:
- Create unique transaction_id where 1 transaction can have N rows
- CONCATINATE from 6 txt files into 1 csv
- DROP columns
- STANDARDIZE column names
- INVALID VALUES
- Convert DATA TYPES
- Deal with NULLS
- Deal with DUPLICATES
- Deal with OUTLIERS
- Save CLEAN CSV
```

<style>
h3 {
    margin-top: 0px;
    margin-bottom: 0px;
    font-weight: bold;
}
</style>

### 📂 IMPORTs

In [1]:
# import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
import seaborn as sns

plt.style.use('ggplot')
pd.set_option('display.max_columns', 200) # to display all columns in the dataframe
print("✅ Libraries imported successfully")

✅ Libraries imported successfully


<hr>

## 0️⃣ CONCATINATE 

<style>
h1 {
    text-align: center;
    color: black;
    font-weight: bold;
}
</style>

<style>
h2 {
    text-align: left;
    color: black;
    font-weight: bold;
}
</style>

<style>
h3 {
    text-align: left;
    color: black;
    font-weight: bold;
}
</style>

<style>
h4 {
    text-align: left;
    color: black;
    font-weight: bold;
}
</style>
<hr>

### `No disposition` **UNIQUE VALUES**

- load each txt file into a pandas dataframe
- then convert each to a CSV and save to output directory
- using chunk processing to handle large files
- We need to check each file unique values of column "No disposition"
- We need to create a unique identifier for each transaction where:
1 transaction can have N rows


In [ ]:
from pathlib import Path
import pandas as pd

RAW_DIR = Path("../data/raw")
OUTPUT_DIR = Path("../data/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FILES = [
    "ValeursFoncieres-2020-S2.txt",
    "ValeursFoncieres-2021.txt",
    "ValeursFoncieres-2022.txt",
    "ValeursFoncieres-2023.txt",
    "ValeursFoncieres-2024.txt",
    "ValeursFoncieres-2025-S1.txt",
]

COLUMN = "No disposition"
CHUNKSIZE = 500_000

for file in FILES:
    input_path = RAW_DIR / file
    output_path = OUTPUT_DIR / file.replace(".txt", ".csv")

    unique_values = set()
    first_chunk = True

    for chunk in pd.read_csv(
        input_path,
        sep="|",
        dtype=str,
        chunksize=CHUNKSIZE,
        encoding="utf-8",
        low_memory=False
    ):
        if COLUMN in chunk.columns:
            unique_values.update(chunk[COLUMN].dropna().unique())

        chunk.to_csv(
            output_path,
            index=False,
            mode="w" if first_chunk else "a",
            header=first_chunk
        )

        first_chunk = False

    print(f"\n{file}")
    print(f"Unique values in '{COLUMN}':")
    print(sorted(unique_values))


ValeursFoncieres-2020-S2.txt
Unique values in 'No disposition':
['000001', '000002', '000003', '000004', '000005', '000006', '000007', '000008', '000009', '000010', '000011', '000012', '000013', '000014', '000015', '000016', '000017', '000018', '000019', '000020', '000021', '000022', '000023', '000024', '000025', '000026', '000027', '000028', '000029', '000030', '000031', '000032', '000033', '000034', '000035', '000036', '000037', '000038', '000039', '000040', '000041', '000042', '000043', '000044', '000045', '000046', '000047', '000048', '000049', '000050', '000051', '000052', '000053', '000054', '000055', '000056', '000057', '000058', '000059', '000060', '000061', '000062', '000063', '000064', '000065', '000066', '000067', '000068', '000069', '000070', '000071', '000072', '000073', '000074', '000075', '000076', '000077', '000078', '000079', '000080', '000081', '000082', '000083', '000084', '000085', '000086', '000087', '000088', '000089', '000090', '000091', '000092', '000093', '000

### #️⃣ **CREATE unique ID: `transaction_id` & save each file as a CSV** 
**1 transaction → 1 to N rows**

Since No dispostion is reset every year and not quite expressive as an identifier we shall create our own unique transaction_id by combining these columns:

- date: transaction_date ()
- address: insee_code (combine dep_code with com_code)
- transaction_type: "Sale", "Sale in future state of completion", "Sale of unbuilt land","Exchange", "Auction", "Expropriation"
- property_value: use the float number without comma ","
- property_type: "House", "Apartment", "Outbuilding", "Industrial or commercial premises"
- surface_type: "Building", "Land", "Combined"
- property_surface (real_building_surface + land_surface)


**Make sure to drop No disposition in each file** 

In [ ]:
Each file loop:
- convert file to csv 
- generate an ID column transaction_id
- drop No disposition column
- save :
ID_ValeursFoncieres_2020_S2.csv
ID_ValeursFoncieres_2021.csv
ID_ValeursFoncieres_2022.csv
ID_ValeursFoncieres_2023.csv
ID_ValeursFoncieres_2024.csv
ID_ValeursFoncieres_2025_S1.csv
- concat csv files into 1 big file call it CANCAT_ValeursFoncieres.csv

### **FROM 6 FILES to 1 - SAVE**

In [ ]:
save as CONCAT_ValeursFoncieres.csv

<hr>

## 1️⃣ STANDARDIZE NAMES


<style>
h1 {
    text-align: center;
    color: black;
    font-weight: bold;
}
</style>

<style>
h2 {
    text-align: left;
    color: black;
    font-weight: bold;
}
</style>

<style>
h3 {
    text-align: left;
    color: black;
    font-weight: bold;
}
</style>

<style>
h4 {
    text-align: left;
    color: black;
    font-weight: bold;
}
</style>
<hr>

→ WHY?  Ensure consistent naming style; Prevent coding errors and improve readability and collaboration with same standardized column names.

### **SHAPE & HEAD**

In [ ]:
import pandas as pd

FILE = "../data/processed/CONCAT_ValeursFoncieres.csv"
CHUNK_SIZE = 500_000

total_rows = 0
columns = None

for chunk in pd.read_csv(FILE, chunksize=CHUNK_SIZE, dtype=str):
    total_rows += len(chunk)
    if columns is None:
        columns = chunk.columns

print("Shape:", total_rows, "rows and", len(columns), "columns")

df_head = pd.read_csv(FILE, nrows=5, dtype=str)
display(df_head)


Shape: 20102739 rows and 43 columns


,Identifiant de document,Reference document,1 Articles CGI,2 Articles CGI,3 Articles CGI,4 Articles CGI,5 Articles CGI,No disposition,Date mutation,Nature mutation,Valeur fonciere,No voie,B/T/Q,Type de voie,Code voie,Voie,Code postal,Commune,Code departement,Code commune,Prefixe de section,Section,No plan,No Volume,1er lot,Surface Carrez du 1er lot,2eme lot,Surface Carrez du 2eme lot,3eme lot,Surface Carrez du 3eme lot,4eme lot,Surface Carrez du 4eme lot,5eme lot,Surface Carrez du 5eme lot,Nombre de lots,Code type local,Type local,Identifiant local,Surface reelle bati,Nombre pieces principales,Nature culture,Nature culture speciale,Surface terrain
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,000001,01/07/2020,Vente,"31234,16",NaN,NaN,NaN,B064,SAINT JULIEN,1560,SAINT-JULIEN-SUR-REYSSOUZE,01,367,NaN,A,8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,AB,NaN,1192
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,000001,01/07/2020,Vente,"278000,00",NaN,NaN,NaN,B188,A LA PEROUSE,1250,CORVEISSIAT,01,125,NaN,C,509,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,BS,NaN,10092
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,000001,01/07/2020,Vente,"278000,00",NaN,NaN,NaN,B188,A LA PEROUSE,1250,CORVEISSIAT,01,125,NaN,C,510,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,L,NaN,4570
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,000001,01/07/2020,Vente,"278000,00",NaN,NaN,NaN,B079,AUX COMMUNS,1250,CORVEISSIAT,01,125,NaN,ZL,96,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,BS,NaN,5750
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,000001,01/07/2020,Vente,"278000,00",NaN,NaN,NaN,B033,EN COMBARNAUD,1250,SIMANDRE-SUR-SURAN,01,408,NaN,A,14,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,BT,NaN,648170


### **INFO (sample)**

In [ ]:
FILE = "../data/processed/CONCAT_ValeursFoncieres.csv"
df_sample = pd.read_csv(FILE, nrows=100_000, dtype=str)
df_sample.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 35 columns):
 #   Column                      Non-Null Count   Dtype 
---  ------                      --------------   ----- 
 0   No disposition              100000 non-null  object
 1   Date mutation               100000 non-null  object
 2   Nature mutation             100000 non-null  object
 3   Valeur fonciere             99092 non-null   object
 4   No voie                     59909 non-null   object
 5   B/T/Q                       3940 non-null    object
 6   Type de voie                56183 non-null   object
 7   Code voie                   98826 non-null   object
 8   Voie                        98826 non-null   object
 9   Code postal                 98826 non-null   object
 10  Commune                     100000 non-null  object
 11  Code departement            100000 non-null  object
 12  Code commune                100000 non-null  object
 13  Prefixe de section          18

### **INFO**

In [ ]:
import pandas as pd

FILE = "../data/processed/CONCAT_ValeursFoncieres.csv"
CHUNK_SIZE = 500_000

total_rows = 0
columns = None
non_null_counts = None
dtypes = None

for chunk in pd.read_csv(FILE, chunksize=CHUNK_SIZE, dtype=str):
    if columns is None:
        columns = chunk.columns.tolist()
        non_null_counts = {col: 0 for col in columns}
        dtypes = chunk.dtypes.astype(str).to_dict()

    total_rows += len(chunk)

    for col in columns:
        non_null_counts[col] += chunk[col].notna().sum()

# Build info-style summary
info_df = pd.DataFrame({
    "column": columns,
    "non_null_count": [non_null_counts[col] for col in columns],
    "null_count": [total_rows - non_null_counts[col] for col in columns],
    "dtype": [dtypes[col] for col in columns]
})

print(f"RangeIndex: {total_rows} entries, 0 to {total_rows - 1}")
print(f"Data columns (total {len(columns)} columns):")
display(info_df)
print(f"dtypes: {info_df['dtype'].value_counts().to_dict()}")
print(f"memory usage: not computed (chunked mode)")

RangeIndex: 20102739 entries, 0 to 20102738
Data columns (total 35 columns):


,column,non_null_count,null_count,dtype
0,No disposition,20102739,0,object
1,Date mutation,20102739,0,object
2,Nature mutation,20102739,0,object
3,Valeur fonciere,19908349,194390,object
4,No voie,12628228,7474511,object
5,B/T/Q,905080,19197659,object
6,Type de voie,12160680,7942059,object
7,Code voie,19952257,150482,object
8,Voie,19951476,151263,object
9,Code postal,19951139,151600,object


dtypes: {'object': 35}
memory usage: not computed (chunked mode)


### **UNIQUE VALUES & NULLS**

In [ ]:
from collections import defaultdict
import pandas as pd

FILE = "../data/processed/CONCAT_ValeursFoncieres.csv"

unique_values = defaultdict(set)
null_counts = defaultdict(int)
total_rows = 0

for chunk in pd.read_csv(
    FILE,
    chunksize=500_000,
    dtype=str,          # ✅ avoids mixed types warning
    low_memory=False
):
    total_rows += len(chunk)

    for col in chunk.columns:
        # ✅ Count nulls (DO NOT drop them)
        null_counts[col] += chunk[col].isna().sum()

        # ✅ Collect only non-null values (nulls are still preserved in dataset)
        unique_values[col].update(
            chunk.loc[chunk[col].notna(), col].unique()
        )

# --------------------------------------------------
# Display results
# --------------------------------------------------
print(f"\nTotal rows processed: {total_rows:,}")
print(100 * "-")

for col in unique_values:
    values = list(unique_values[col])
    null_pct = (null_counts[col] / total_rows) * 100

    print(f"➡️ {col}:")
    print(f"Unique values count: {len(values)}")
    print(f"Null count: {null_counts[col]} ({null_pct:.2f}%)")
    print(f"Sample values: {values[:120]}")
    print(100 * "-")


Total rows processed: 20,102,739
----------------------------------------------------------------------------------------------------
➡️ No disposition:
Unique values count: 1204
Null count: 0 (0.00%)
Sample values: ['000576', '000189', '000145', '000546', '000951', '001152', '001186', '000358', '001194', '000797', '000915', '000261', '000925', '001052', '000988', '000115', '001043', '000855', '000422', '000830', '000232', '000411', '001132', '001148', '000440', '001011', '000494', '000462', '000475', '001021', '001154', '000183', '000312', '001017', '001110', '000852', '000511', '000645', '000918', '000629', '000398', '000361', '001074', '000420', '000141', '000164', '000085', '000697', '000126', '000308', '000672', '001055', '000755', '000283', '000470', '000586', '000043', '000432', '001108', '000122', '000153', '000493', '001121', '000055', '000031', '000223', '000143', '000229', '000694', '000707', '000714', '001030', '000776', '000949', '000712', '000483', '000167', '000341', '0

### **DATASET TABLE V1 (Original vs Standard Names)**

| #  | Original Column Name           | Standardized            | Description                                  |
| -- | ------------------------------ | ----------------------- | -------------------------------------------- |
| 0  | **No disposition**             | `transaction_id`        | Unique identifier of the transaction         |
| 1  | **Date mutation**              | `transaction_date`      | Date when the property was sold              |
| 2  | **Nature mutation**            | `transaction_type`      | Nature of transaction (sale, exchange, etc.) |
| 3  | **Valeur fonciere**            | `property_value`        | Transaction price in euros                   |
| 4  | **No voie**                    | `street_number`         | Street number of the property                |
| 5  | **B/T/Q**                      | `btq_code`              | Building / Type / Quarter code               |
| 6  | **Type de voie**               | `street_type`           | Type of street (Rue, Avenue, etc.)           |
| 7  | **Code voie**                  | `street_id`             | Street identifier code                       |
| 8  | **Voie**                       | `street_name`           | Full street name                             |
| 9  | **Code postal**                | `postal_code`           | Postal code (categorical, not numeric)       |
| 10 | **Commune**                    | `com_name`              | Commune name                                 |
| 11 | **Code departement**           | `dep_code`              | Department code: DD                          |
| 12 | **Code commune**               | `com_code`              | Commune code: CCC                            |
| 13 | **Prefixe de section**         | `section_prefix`        | Section prefix (cadastral division)          |
| 14 | **Section**                    | `section`               | Cadastral section identifier                 |
| 15 | **No plan**                    | `plot_number`           | Parcel/plot number                           |
| 16 | **No Volume**                  | `volume_number`         | Volume number (rarely used cadastral info)   |
| 17 | **1er lot**                    | `lot_1`                 | Identifier of the first lot                  |
| 18 | **Surface Carrez du 1er lot**  | `lot_1_surface`         | Carrez surface of the first lot (m²)         |
| 19 | **2eme lot**                   | `lot_2`                 | Identifier of the second lot                 |
| 20 | **Surface Carrez du 2eme lot** | `lot_2_surface`         | Carrez surface of the second lot (m²)        |
| 21 | **3eme lot**                   | `lot_3`                 | Identifier of the third lot                  |
| 22 | **Surface Carrez du 3eme lot** | `lot_3_surface`         | Carrez surface of the third lot (m²)         |
| 23 | **4eme lot**                   | `lot_4`                 | Identifier of the fourth lot                 |
| 24 | **Surface Carrez du 4eme lot** | `lot_4_surface`         | Carrez surface of the fourth lot (m²)        |
| 25 | **5eme lot**                   | `lot_5`                 | Identifier of the fifth lot                  |
| 26 | **Surface Carrez du 5eme lot** | `lot_5_surface`         | Carrez surface of the fifth lot (m²)         |
| 27 | **Nombre de lots**             | `lots_count`            | Number of lots in the transaction            |
| 28 | **Code type local**            | `property_type_code`    | Code for property type                       |
| 29 | **Type local**                 | `property_type`         | Property type (Apartment, House, etc.)       |
| 30 | **Surface reelle bati**        | `building_real_surface` | Built area in square meters                  |
| 31 | **Nombre pieces principales**  | `main_rooms_count`      | Number of main rooms                         |
| 32 | **Nature culture**             | `land_nature`           | Type of land usage                           |
| 33 | **Nature culture speciale**    | `land_nature_special`   | Specific land usage classification           |
| 34 | **Surface terrain**            | `land_surface`          | Land area in square meters                   |


### **RENAME COLUMNS**

In [ ]:
from pathlib import Path
import pandas as pd

INPUT_FILE = Path("../data/processed/CONCAT_ValeursFoncieres.csv")
OUTPUT_FILE = Path("../data/processed/STANDARD_ValeursFoncieres.csv")

CHUNK_SIZE = 500_000

# ---------------------------------------------
# Define the renaming dictionary
# ---------------------------------------------
rename_columns = {
    "No disposition": "transaction_id",
    "Date mutation": "transaction_date",
    "Nature mutation": "transaction_type",
    "Valeur fonciere": "property_value",
    "No voie": "street_number",
    "B/T/Q": "btq_code",
    "Type de voie": "street_type",
    "Code voie": "street_id",
    "Voie": "street_name",
    "Code postal": "postal_code",
    "Commune": "com_name",
    "Code departement": "dep_code",
    "Code commune": "com_code",
    "Prefixe de section": "section_prefix",
    "Section": "section",
    "No plan": "plot_number",
    "No Volume": "volume_number",

    # Lot identifier and surface
    "1er lot": "lot_1",
    "Surface Carrez du 1er lot": "lot_1_surface",
    "2eme lot": "lot_2",
    "Surface Carrez du 2eme lot": "lot_2_surface",
    "3eme lot": "lot_3",
    "Surface Carrez du 3eme lot": "lot_3_surface",
    "4eme lot": "lot_4",
    "Surface Carrez du 4eme lot": "lot_4_surface",
    "5eme lot": "lot_5",
    "Surface Carrez du 5eme lot": "lot_5_surface",

    "Nombre de lots": "lots_count",

    "Code type local": "property_type_code",
    "Type local": "property_type",
    "Surface reelle bati": "building_real_surface",
    "Nombre pieces principales": "main_rooms_count",
    "Nature culture": "land_nature",
    "Nature culture speciale": "land_nature_special",
    "Surface terrain": "land_surface",
}

def rename_columns_chunked(
    input_file: Path,
    output_file: Path,
    rename_dict: dict,
    chunk_size: int = 50_000
) -> None:
    output_file.parent.mkdir(parents=True, exist_ok=True)

    first_write = True
    total_rows = 0

    for chunk_idx, chunk in enumerate(
        pd.read_csv(input_file, chunksize=chunk_size, dtype=str),
        start=1
    ):
        chunk = chunk.rename(columns=rename_dict)

        chunk.to_csv(
            output_file,
            mode="w" if first_write else "a",
            index=False,
            header=first_write,
            encoding="utf-8"
        )

        first_write = False
        total_rows += len(chunk)

        print(f"Chunk {chunk_idx} written -> {len(chunk):,} rows | total: {total_rows:,}")

    print("\n✅ Done! Standardized columns")
    print(f"Saved file: {output_file}")
    print(f"Shape: {total_rows:,} rows and {len(rename_dict)} columns")

rename_columns_chunked(INPUT_FILE, OUTPUT_FILE, rename_columns, CHUNK_SIZE)

Chunk 1 written -> 500,000 rows | total: 500,000
Chunk 2 written -> 500,000 rows | total: 1,000,000
Chunk 3 written -> 500,000 rows | total: 1,500,000
Chunk 4 written -> 500,000 rows | total: 2,000,000
Chunk 5 written -> 500,000 rows | total: 2,500,000
Chunk 6 written -> 500,000 rows | total: 3,000,000
Chunk 7 written -> 500,000 rows | total: 3,500,000
Chunk 8 written -> 500,000 rows | total: 4,000,000
Chunk 9 written -> 500,000 rows | total: 4,500,000
Chunk 10 written -> 500,000 rows | total: 5,000,000
Chunk 11 written -> 500,000 rows | total: 5,500,000
Chunk 12 written -> 500,000 rows | total: 6,000,000
Chunk 13 written -> 500,000 rows | total: 6,500,000
Chunk 14 written -> 500,000 rows | total: 7,000,000
Chunk 15 written -> 500,000 rows | total: 7,500,000
Chunk 16 written -> 500,000 rows | total: 8,000,000
Chunk 17 written -> 500,000 rows | total: 8,500,000
Chunk 18 written -> 500,000 rows | total: 9,000,000
Chunk 19 written -> 500,000 rows | total: 9,500,000
Chunk 20 written -> 500

<hr>

## 2️⃣ INVALID VALUES


<style>
h1 {
    text-align: center;
    color: black;
    font-weight: bold;
}
</style>

<style>
h2 {
    text-align: left;
    color: black;
    font-weight: bold;
}
</style>

<style>
h3 {
    text-align: left;
    color: black;
    font-weight: bold;
}
</style>

<style>
h4 {
    text-align: left;
    color: black;
    font-weight: bold;
}
</style>
<hr>

→ WHY? Ensure logical consistency of values in each column

### **SHAPE & HEAD**

In [7]:
import pandas as pd

FILE = "../data/processed/STANDARD_ValeursFoncieres.csv"
CHUNK_SIZE = 500_000

total_rows = 0
columns = None

for chunk in pd.read_csv(FILE, chunksize=CHUNK_SIZE, dtype=str):
    total_rows += len(chunk)
    if columns is None:
        columns = chunk.columns

print("Shape:", total_rows, "rows and", len(columns), "columns")

df_head = pd.read_csv(FILE, nrows=5, dtype=str)
display(df_head)


Shape: 20102739 rows and 35 columns


,transaction_id,transaction_date,transaction_type,property_value,street_number,btq_code,street_type,street_id,street_name,postal_code,com_name,dep_code,com_code,section_prefix,section,plot_number,volume_number,lot_1,lot_1_surface,lot_2,lot_2_surface,lot_3,lot_3_surface,lot_4,lot_4_surface,lot_5,lot_5_surface,lots_count,property_type_code,property_type,building_real_surface,main_rooms_count,land_nature,land_nature_special,land_surface
0,000001,01/07/2020,Vente,"31234,16",NaN,NaN,NaN,B064,SAINT JULIEN,1560,SAINT-JULIEN-SUR-REYSSOUZE,01,367,NaN,A,8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,AB,NaN,1192
1,000001,01/07/2020,Vente,"278000,00",NaN,NaN,NaN,B188,A LA PEROUSE,1250,CORVEISSIAT,01,125,NaN,C,509,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,BS,NaN,10092
2,000001,01/07/2020,Vente,"278000,00",NaN,NaN,NaN,B188,A LA PEROUSE,1250,CORVEISSIAT,01,125,NaN,C,510,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,L,NaN,4570
3,000001,01/07/2020,Vente,"278000,00",NaN,NaN,NaN,B079,AUX COMMUNS,1250,CORVEISSIAT,01,125,NaN,ZL,96,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,BS,NaN,5750
4,000001,01/07/2020,Vente,"278000,00",NaN,NaN,NaN,B033,EN COMBARNAUD,1250,SIMANDRE-SUR-SURAN,01,408,NaN,A,14,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,BT,NaN,648170


### **INFO**

In [8]:
import pandas as pd

FILE = "../data/processed/STANDARD_ValeursFoncieres.csv"
CHUNK_SIZE = 500_000

total_rows = 0
columns = None
non_null_counts = None
dtypes = None

for chunk in pd.read_csv(FILE, chunksize=CHUNK_SIZE, dtype=str):
    if columns is None:
        columns = chunk.columns.tolist()
        non_null_counts = {col: 0 for col in columns}
        dtypes = chunk.dtypes.astype(str).to_dict()

    total_rows += len(chunk)

    for col in columns:
        non_null_counts[col] += chunk[col].notna().sum()

# Build info-style summary
info_df = pd.DataFrame({
    "column": columns,
    "non_null_count": [non_null_counts[col] for col in columns],
    "null_count": [total_rows - non_null_counts[col] for col in columns],
    "dtype": [dtypes[col] for col in columns]
})

print(f"RangeIndex: {total_rows} entries, 0 to {total_rows - 1}")
print(f"Data columns (total {len(columns)} columns):")
display(info_df)
print(f"dtypes: {info_df['dtype'].value_counts().to_dict()}")
print(f"memory usage: not computed (chunked mode)")

RangeIndex: 20102739 entries, 0 to 20102738
Data columns (total 35 columns):


,column,non_null_count,null_count,dtype
0,transaction_id,20102739,0,object
1,transaction_date,20102739,0,object
2,transaction_type,20102739,0,object
3,property_value,19908349,194390,object
4,street_number,12628228,7474511,object
5,btq_code,905080,19197659,object
6,street_type,12160680,7942059,object
7,street_id,19952257,150482,object
8,street_name,19951476,151263,object
9,postal_code,19951139,151600,object


dtypes: {'object': 35}
memory usage: not computed (chunked mode)


### **UNIQUE VALUES & NULLS**

In [9]:
from collections import defaultdict
import pandas as pd

FILE = "../data/processed/STANDARD_ValeursFoncieres.csv"

unique_values = defaultdict(set)
null_counts = defaultdict(int)
total_rows = 0

for chunk in pd.read_csv(
    FILE,
    chunksize=500_000,
    dtype=str,          # ✅ avoids mixed types warning
    low_memory=False
):
    total_rows += len(chunk)

    for col in chunk.columns:
        # ✅ Count nulls (DO NOT drop them)
        null_counts[col] += chunk[col].isna().sum()

        # ✅ Collect only non-null values (nulls are still preserved in dataset)
        unique_values[col].update(
            chunk.loc[chunk[col].notna(), col].unique()
        )

# --------------------------------------------------
# Display results
# --------------------------------------------------
print(f"\nTotal rows processed: {total_rows:,}")
print(100 * "-")

for col in unique_values:
    values = list(unique_values[col])
    null_pct = (null_counts[col] / total_rows) * 100

    print(f"➡️ {col}:")
    print(f"Unique values count: {len(values)}")
    print(f"Null count: {null_counts[col]} ({null_pct:.2f}%)")
    print(f"Sample values: {values[:120]}")
    print(100 * "-")


Total rows processed: 20,102,739
----------------------------------------------------------------------------------------------------
➡️ transaction_id:
Unique values count: 1204
Null count: 0 (0.00%)
Sample values: ['000576', '000189', '000145', '000546', '000951', '001152', '001186', '000358', '001194', '000797', '000915', '000261', '000925', '001052', '000988', '000115', '001043', '000855', '000422', '000830', '000232', '000411', '001132', '001148', '000440', '001011', '000494', '000462', '000475', '001021', '001154', '000183', '000312', '001017', '001110', '000852', '000511', '000645', '000918', '000629', '000398', '000361', '001074', '000420', '000141', '000164', '000085', '000697', '000126', '000308', '000672', '001055', '000755', '000283', '000470', '000586', '000043', '000432', '001108', '000122', '000153', '000493', '001121', '000055', '000031', '000223', '000143', '000229', '000694', '000707', '000714', '001030', '000776', '000949', '000712', '000483', '000167', '000341', '0

### **Deal with INVALID VALUES**

In [10]:
from pathlib import Path
import pandas as pd

INPUT_FILE = Path("../data/processed/STANDARD_ValeursFoncieres.csv")
OUTPUT_FILE = Path("../data/processed/INVALIDS_ValeursFoncieres.csv")

CHUNK_SIZE = 500_000

date_cols = [
    "transaction_date",
]

numeric_float_cols = [
    "property_value",
    "building_real_surface",
    "land_surface",
    "lot_1_surface",
    "lot_2_surface",
    "lot_3_surface",
    "lot_4_surface",
    "lot_5_surface",
]

numeric_int_cols = [
    "lots_count",
    "main_rooms_count",
]

categorical_cols = [
    "transaction_id",
    "transaction_type",
    "street_number",
    "btq_code",
    "street_type",
    "street_id",
    "street_name",
    "postal_code",
    "com_name",
    "dep_code",
    "com_code",
    "section_prefix",
    "section",
    "plot_number",
    "volume_number",
    "property_type_code",
    "property_type",
    "land_nature",
    "land_nature_special",
    "lot_1",
    "lot_2",
    "lot_3",
    "lot_4",
    "lot_5",
]

# STANDARDIZE FAKE NULLS to NaN
missing_markers = {
    "MISSING", "", " ", "NA", "N/A", "None", "null", "NULL", "nan", "NaN"
}


def standardize_nulls(chunk: pd.DataFrame) -> pd.DataFrame:
    for col in chunk.columns:
        if chunk[col].dtype == "object" or str(chunk[col].dtype) == "string":
            chunk[col] = chunk[col].astype("string").str.strip()
            chunk[col] = chunk[col].replace(missing_markers, pd.NA)
    return chunk


def clean_date_columns(chunk: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    for col in cols:
        if col in chunk.columns:
            chunk[col] = pd.to_datetime(
                chunk[col],
                format="%d/%m/%Y",
                errors="coerce"
            )
    return chunk


def clean_float_columns(chunk: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    for col in cols:
        if col in chunk.columns:
            chunk[col] = (
                chunk[col]
                .astype("string")
                .str.replace(",", ".", regex=False)
                .str.strip()
            )
            chunk[col] = pd.to_numeric(chunk[col], errors="coerce")
    return chunk


def clean_int_columns(chunk: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    for col in cols:
        if col in chunk.columns:
            chunk[col] = (
                chunk[col]
                .astype("string")
                .str.strip()
            )
            chunk[col] = pd.to_numeric(chunk[col], errors="coerce").astype("Int64")
    return chunk


def clean_categorical_columns(chunk: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    for col in cols:
        if col in chunk.columns:
            chunk[col] = chunk[col].astype("string").str.strip()
    return chunk


def apply_custom_invalid_rules(chunk: pd.DataFrame) -> pd.DataFrame:
    # transaction_type: translate and standardize
    if "transaction_type" in chunk.columns:
        chunk["transaction_type"] = chunk["transaction_type"].replace({
            "Vente": "Sale",
            "Vente en l'état futur d'achèvement": "Sale in future state of completion",
            "Vente terrain à bâtir": "Sale of unbuilt land",
            "Echange": "Exchange",
            "Adjudication": "Auction",
            "Expropriation": "Expropriation",
        })

    # property_type: translate actual DVF categories seen in your data
    if "property_type" in chunk.columns:
        chunk["property_type"] = chunk["property_type"].replace({
            "Maison": "House",
            "Appartement": "Apartment",
            "Dépendance": "Outbuilding",
            "Local industriel. commercial ou assimilé": "Industrial or commercial premises",
        })

    # btq_code: clean invalid symbols and standardize lowercase b
    if "btq_code" in chunk.columns:
        chunk["btq_code"] = chunk["btq_code"].replace(["-", ".", "*"], pd.NA)
        chunk["btq_code"] = chunk["btq_code"].replace({"b": "B"})

    # building_real_surface and main_rooms_count:
    # use 0 when missing at this invalid-handling stage
    if "building_real_surface" in chunk.columns:
        chunk["building_real_surface"] = chunk["building_real_surface"].fillna(0.0)

    if "main_rooms_count" in chunk.columns:
        chunk["main_rooms_count"] = chunk["main_rooms_count"].fillna(0).astype("Int64")

    # create surface_type
    if {"building_real_surface", "land_surface"}.issubset(chunk.columns):
        b = chunk["building_real_surface"].fillna(0)
        l = chunk["land_surface"].fillna(0)

        chunk["surface_type"] = pd.NA
        chunk.loc[(b > 0) & (l == 0), "surface_type"] = "building"
        chunk.loc[(b == 0) & (l > 0), "surface_type"] = "land"
        chunk.loc[(b > 0) & (l > 0), "surface_type"] = "combined"

    return chunk


def clean_chunk(chunk: pd.DataFrame) -> pd.DataFrame:
    chunk = standardize_nulls(chunk)
    chunk = clean_date_columns(chunk, date_cols)
    chunk = clean_float_columns(chunk, numeric_float_cols)
    chunk = clean_int_columns(chunk, numeric_int_cols)
    chunk = clean_categorical_columns(chunk, categorical_cols)
    chunk = apply_custom_invalid_rules(chunk)
    return chunk


def clean_file_chunked(input_file: Path, output_file: Path, chunk_size: int = 50_000) -> None:
    output_file.parent.mkdir(parents=True, exist_ok=True)

    first_write = True
    total_rows = 0

    for chunk_idx, chunk in enumerate(
        pd.read_csv(input_file, chunksize=chunk_size, dtype=str, low_memory=False),
        start=1
    ):
        chunk = clean_chunk(chunk)

        chunk.to_csv(
            output_file,
            mode="w" if first_write else "a",
            index=False,
            header=first_write,
            encoding="utf-8"
        )

        first_write = False
        total_rows += len(chunk)

        print(f"Chunk {chunk_idx} written -> {len(chunk):,} rows | total: {total_rows:,}")

    print("\n✅ Done! Handled invalid values")
    print(f"Saved file: {output_file}")
    print(f"Total rows written: {total_rows:,}")


clean_file_chunked(INPUT_FILE, OUTPUT_FILE, CHUNK_SIZE)

Chunk 1 written -> 500,000 rows | total: 500,000
Chunk 2 written -> 500,000 rows | total: 1,000,000
Chunk 3 written -> 500,000 rows | total: 1,500,000
Chunk 4 written -> 500,000 rows | total: 2,000,000
Chunk 5 written -> 500,000 rows | total: 2,500,000
Chunk 6 written -> 500,000 rows | total: 3,000,000
Chunk 7 written -> 500,000 rows | total: 3,500,000
Chunk 8 written -> 500,000 rows | total: 4,000,000
Chunk 9 written -> 500,000 rows | total: 4,500,000
Chunk 10 written -> 500,000 rows | total: 5,000,000
Chunk 11 written -> 500,000 rows | total: 5,500,000
Chunk 12 written -> 500,000 rows | total: 6,000,000
Chunk 13 written -> 500,000 rows | total: 6,500,000
Chunk 14 written -> 500,000 rows | total: 7,000,000
Chunk 15 written -> 500,000 rows | total: 7,500,000
Chunk 16 written -> 500,000 rows | total: 8,000,000
Chunk 17 written -> 500,000 rows | total: 8,500,000
Chunk 18 written -> 500,000 rows | total: 9,000,000
Chunk 19 written -> 500,000 rows | total: 9,500,000
Chunk 20 written -> 500

### **DATASET TABLE V2**

| #  | Original Column Name           | Standardized            | Description                                       | Data Type | Type                 |
| -- | ------------------------------ | ----------------------- | ------------------------------------------------- | --------- | -------------------- |
| 0  | **No disposition**             | `transaction_id`        | Unique identifier of the transaction              | string    | Categorical          |
| 1  | **Date mutation**              | `transaction_date`      | Date when the property was sold                   | string    | Temporal             |
| 2  | **Nature mutation**            | `transaction_type`      | Nature of transaction (sale, exchange, etc.)      | string    | Categorical          |
| 3  | **Valeur fonciere**            | `property_value`        | Transaction price in euros                        | float     | Numeric (**TARGET**) |
| 4  | **No voie**                    | `street_number`         | Street number of the property                     | string    | Categorical          |
| 5  | **B/T/Q**                      | `btq_code`              | Building / Type / Quarter code                    | string    | Categorical          |
| 6  | **Type de voie**               | `street_type`           | Type of street (Rue, Avenue, etc.)                | string    | Categorical          |
| 7  | **Code voie**                  | `street_id`             | Street identifier code                            | string    | Categorical          |
| 8  | **Voie**                       | `street_name`           | Full street name                                  | string    | Categorical          |
| 9  | **Code postal**                | `postal_code`           | Postal code                                       | string    | Categorical          |
| 10 | **Commune**                    | `com_name`              | Commune name                                      | string    | Categorical          |
| 11 | **Code departement**           | `dep_code`              | Department code                                   | string    | Categorical          |
| 12 | **Code commune**               | `com_code`              | Commune code                                      | string    | Categorical          |
| 13 | **Prefixe de section**         | `section_prefix`        | Section prefix used in cadastral identification   | string    | Categorical          |
| 14 | **Section**                    | `section`               | Cadastral section identifier                      | string    | Categorical          |
| 15 | **No plan**                    | `plot_number`           | Parcel / plot number                              | string    | Categorical          |
| 16 | **No Volume**                  | `volume_number`         | Volume number for special cadastral cases         | string    | Categorical          |
| 17 | **1er lot**                    | `lot_1`                 | Identifier of the first lot                       | string    | Categorical          |
| 18 | **Surface Carrez du 1er lot**  | `lot_1_carrez_surface`  | Carrez surface of the first lot in square meters  | float     | Numeric              |
| 19 | **2eme lot**                   | `lot_2`                 | Identifier of the second lot                      | string    | Categorical          |
| 20 | **Surface Carrez du 2eme lot** | `lot_2_carrez_surface`  | Carrez surface of the second lot in square meters | float     | Numeric              |
| 21 | **3eme lot**                   | `lot_3`                 | Identifier of the third lot                       | string    | Categorical          |
| 22 | **Surface Carrez du 3eme lot** | `lot_3_carrez_surface`  | Carrez surface of the third lot in square meters  | float     | Numeric              |
| 23 | **4eme lot**                   | `lot_4`                 | Identifier of the fourth lot                      | string    | Categorical          |
| 24 | **Surface Carrez du 4eme lot** | `lot_4_carrez_surface`  | Carrez surface of the fourth lot in square meters | float     | Numeric              |
| 25 | **5eme lot**                   | `lot_5`                 | Identifier of the fifth lot                       | string    | Categorical          |
| 26 | **Surface Carrez du 5eme lot** | `lot_5_carrez_surface`  | Carrez surface of the fifth lot in square meters  | float     | Numeric              |
| 27 | **Nombre de lots**             | `lots_count`            | Number of lots in the transaction                 | integer   | Numeric              |
| 28 | **Code type local**            | `property_type_code`    | Code for property type                            | string    | Categorical          |
| 29 | **Type local**                 | `property_type`         | Property type (Apartment, House, etc.)            | string    | Categorical          |
| 30 | **Surface reelle bati**        | `building_real_surface` | Built area in square meters                       | float     | Numeric              |
| 31 | **Nombre pieces principales**  | `main_rooms_count`      | Number of main rooms                              | integer   | Numeric              |
| 32 | **Nature culture**             | `land_nature`           | General type of land usage                        | string    | Categorical          |
| 33 | **Nature culture speciale**    | `land_nature_special`   | Specific type of land usage                       | string    | Categorical          |
| 34 | **Surface terrain**            | `land_surface`          | Land area in square meters                        | float     | Numeric              |
| 35 | *(derived)*                    | `surface_type`          | Type of surface: building / land / combined       | string    | Derived Categorical  |


---
###  **CREATE `df_com_dep`**

from `df_com` :
- ✔ COMTYPE → commune type
- ✔ COM → commune INSEE code
- ✔ NCCENR → official commune names in MAJ & "-" & ACCENT
- ✔ DEP → department code

from `df_dep` :
 - ✔ DEP → department code
 - ✔ NCCENR → official department names NAMES

combine: add dep_name using DEP

rename columns

save csv

#### **CREATE `df_com`**
- create df_com with select columns from v_communes_2026.csv
	- COM
	- TYPECOM
	- NCCENR (uppercase + encoding & decoding)
	- DEP
- rename df_com columns
	- insee_code = COM
	- com_type = TYPECOM 
	- com_name = NCCENR
	- dep_code = DEP


In [ ]:
# load raw communes dataset
try:
    df_com = pd.read_csv("../data/raw/v_commune_2026.csv", sep=",", encoding="utf-8")
except UnicodeDecodeError:
    df_com = pd.read_csv("../data/raw/v_commune_2026.csv", sep=",", encoding="latin-1")

# rename df_com columns
df_com.rename(columns={
    "COM": "insee_code",
    "TYPECOM": "com_type",
    "NCCENR": "com_name",
    "DEP": "dep_code"
}, inplace=True)

# uppercase + encoding & decoding of com_name
df_com["com_name"] = (
    df_com["com_name"]
    .astype(str)
    .str.upper()
)

# drop other columns except insee_code, com_type, com_name, dep_code
df_com = df_com[["insee_code", "com_type", "com_name", "dep_code"]]

# display df_com
print("Communes Dataset Shape:", df_com.shape[0], "rows and", df_com.shape[1], "columns")
display(df_com.head())
print(100 * "-")
# display unique values of com_name
print("Unique Values of `com_name`:")
display(df_com["com_name"].unique())

# save df_com in a new CSV file
df_com.to_csv("../data/processed/COM_2026.csv", index=False)

#### **CREATE `df_dep`**

In [ ]:
# load raw departement dataset
try:
    df_dep = pd.read_csv("../data/raw/v_departement_2026.csv", sep=",", encoding="utf-8")
except UnicodeDecodeError:
    df_dep = pd.read_csv("../data/raw/v_departement_2026.csv", sep=",", encoding="latin-1")

# rename df_dep columns
df_dep.rename(columns={
    "DEP": "dep_code",
    "NCCENR": "dep_name"
}, inplace=True)

# uppercase + encoding & decoding of dep_name
df_dep["dep_name"] = (
    df_dep["dep_name"]
    .astype(str)
    .str.upper()
)

# drop other columns except dep_code, dep_name
df_dep = df_dep[["dep_code", "dep_name"]]

# display df_dep
print("Departements Dataset Shape:", df_dep.shape[0], "rows and", df_dep.shape[1], "columns")
display(df_dep.head())
print(100 * "-")
# display unique values of dep_name
print("Unique Values of `dep_name`:")
display(df_dep["dep_name"].unique())

# save df_dep in a new CSV file
df_dep.to_csv("../data/processed/DEP_2026.csv", index=False)

#### **USE `com_name` column to COMBINE and create `COM_DEP_2026.CSV`**

In [ ]:
# create df_com_dep by merging df_com and df_dep on dep_code
df_com_dep = pd.merge(df_com, df_dep, on="dep_code", how="left")
# display df_com_dep
print("Combined Communes & Departements Dataset Shape:", df_com_dep.shape[0], "rows and", df_com_dep.shape[1], "columns")
display(df_com_dep.head())
print(100 * "-")
# display unique values of com_name and dep_name in df_com_dep
print("Unique Values of `com_name` in df_com_dep:")
display(df_com_dep["com_name"].unique())
print("Unique Values of `dep_name` in df_com_dep:")
display(df_com_dep["dep_name"].unique())
# save df_com_dep in a new CSV file
df_com_dep.to_csv("../data/processed/COM_DEP_2026.csv", index=False)

<hr>

## 3️⃣ DATA TYPES


<style>
h1 {
    text-align: center;
    color: black;
    font-weight: bold;
}
</style>

<style>
h2 {
    text-align: left;
    color: black;
    font-weight: bold;
}
</style>

<style>
h3 {
    text-align: left;
    color: black;
    font-weight: bold;
}
</style>

<style>
h4 {
    text-align: left;
    color: black;
    font-weight: bold;
}
</style>
<hr>

→  WHY? Enable correct calculations and analysis.

### **SHAPE & HEAD**

In [11]:
import pandas as pd

FILE = "../data/processed/INVALIDS_ValeursFoncieres.csv"
CHUNK_SIZE = 500_000

total_rows = 0
columns = None

for chunk in pd.read_csv(FILE, chunksize=CHUNK_SIZE, dtype=str):
    total_rows += len(chunk)
    if columns is None:
        columns = chunk.columns

print("Shape:", total_rows, "rows and", len(columns), "columns")

df_head = pd.read_csv(FILE, nrows=5, dtype=str)
display(df_head)

Shape: 20102739 rows and 36 columns


,transaction_id,transaction_date,transaction_type,property_value,street_number,btq_code,street_type,street_id,street_name,postal_code,com_name,dep_code,com_code,section_prefix,section,plot_number,volume_number,lot_1,lot_1_surface,lot_2,lot_2_surface,lot_3,lot_3_surface,lot_4,lot_4_surface,lot_5,lot_5_surface,lots_count,property_type_code,property_type,building_real_surface,main_rooms_count,land_nature,land_nature_special,land_surface,surface_type
0,000001,2020-07-01,Sale,31234.16,NaN,NaN,NaN,B064,SAINT JULIEN,1560,SAINT-JULIEN-SUR-REYSSOUZE,01,367,NaN,A,8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,0,0,AB,NaN,1192,land
1,000001,2020-07-01,Sale,278000.0,NaN,NaN,NaN,B188,A LA PEROUSE,1250,CORVEISSIAT,01,125,NaN,C,509,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,0,0,BS,NaN,10092,land
2,000001,2020-07-01,Sale,278000.0,NaN,NaN,NaN,B188,A LA PEROUSE,1250,CORVEISSIAT,01,125,NaN,C,510,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,0,0,L,NaN,4570,land
3,000001,2020-07-01,Sale,278000.0,NaN,NaN,NaN,B079,AUX COMMUNS,1250,CORVEISSIAT,01,125,NaN,ZL,96,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,0,0,BS,NaN,5750,land
4,000001,2020-07-01,Sale,278000.0,NaN,NaN,NaN,B033,EN COMBARNAUD,1250,SIMANDRE-SUR-SURAN,01,408,NaN,A,14,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,0,0,BT,NaN,648170,land


### **UNIQUE VALUES & NULLS**

In [12]:
from collections import defaultdict
import pandas as pd

FILE = "../data/processed/INVALIDS_ValeursFoncieres.csv"

unique_values = defaultdict(set)
null_counts = defaultdict(int)
total_rows = 0

for chunk in pd.read_csv(
    FILE,
    chunksize=500_000,
    dtype=str,          # ✅ avoids mixed types warning
    low_memory=False
):
    total_rows += len(chunk)

    for col in chunk.columns:
        # ✅ Count nulls (DO NOT drop them)
        null_counts[col] += chunk[col].isna().sum()

        # ✅ Collect only non-null values (nulls are still preserved in dataset)
        unique_values[col].update(
            chunk.loc[chunk[col].notna(), col].unique()
        )

# --------------------------------------------------
# Display results
# --------------------------------------------------
print(f"\nTotal rows processed: {total_rows:,}")
print(100 * "-")

for col in unique_values:
    values = list(unique_values[col])
    null_pct = (null_counts[col] / total_rows) * 100

    print(f"➡️ {col}:")
    print(f"Unique values count: {len(values)}")
    print(f"Null count: {null_counts[col]} ({null_pct:.2f}%)")
    print(f"Sample values: {values[:120]}")
    print(100 * "-")


Total rows processed: 20,102,739
----------------------------------------------------------------------------------------------------
➡️ transaction_id:
Unique values count: 1204
Null count: 0 (0.00%)
Sample values: ['000576', '000189', '000145', '000546', '000951', '001152', '001186', '000358', '001194', '000797', '000915', '000261', '000925', '001052', '000988', '000115', '001043', '000855', '000422', '000830', '000232', '000411', '001132', '001148', '000440', '001011', '000494', '000462', '000475', '001021', '001154', '000183', '000312', '001017', '001110', '000852', '000511', '000645', '000918', '000629', '000398', '000361', '001074', '000420', '000141', '000164', '000085', '000697', '000126', '000308', '000672', '001055', '000755', '000283', '000470', '000586', '000043', '000432', '001108', '000122', '000153', '000493', '001121', '000055', '000031', '000223', '000143', '000229', '000694', '000707', '000714', '001030', '000776', '000949', '000712', '000483', '000167', '000341', '0

### **dtype MAPPING**

In [ ]:

dtype_map = {
    # categorical / identifiers
    "transaction_id": "string",
    "transaction_date": "string",  # convert to datetime after loading
    "transaction_type": "string",
    "street_number": "string",
    "btq_code": "string",
    "street_type": "string",
    "street_id": "string",
    "street_name": "string",
    "postal_code": "string",
    "com_name": "string",
    "dep_code": "string",
    "com_code": "string",
    "section_prefix": "string",
    "section": "string",
    "plot_number": "string",
    "volume_number": "string",
    "property_type_code": "string",  # keep as string (categorical)
    "property_type": "string",
    "land_nature": "string",
    "land_nature_special": "string",
    "surface_type": "string",

    # lot identifiers
    "lot_1": "string",
    "lot_2": "string",
    "lot_3": "string",
    "lot_4": "string",
    "lot_5": "string",

    # numeric (use nullable Float64 for consistency)
    "property_value": "Float64",
    "building_real_surface": "Float64",
    "land_surface": "Float64",
    "lot_1_surface": "Float64",
    "lot_2_surface": "Float64",
    "lot_3_surface": "Float64",
    "lot_4_surface": "Float64",
    "lot_5_surface": "Float64",

    # integer (nullable)
    "lots_count": "Int64",
    "main_rooms_count": "Int64",
}

### **CONVERT DATA TYPES**

In [13]:
from pathlib import Path
import pandas as pd

INPUT_FILE = Path("../data/processed/INVALIDS_ValeursFoncieres.csv")
OUTPUT_FILE = Path("../data/processed/TYPES_ValeursFoncieres.csv")

CHUNK_SIZE = 500_000

# Columns by target type
string_cols = [
    "transaction_id",
    "transaction_type",
    "street_number",
    "btq_code",
    "street_type",
    "street_id",
    "street_name",
    "postal_code",
    "com_name",
    "dep_code",
    "com_code",
    "section_prefix",
    "section",
    "plot_number",
    "volume_number",
    "property_type_code",
    "property_type",
    "land_nature",
    "land_nature_special",
    "surface_type",
    "lot_1",
    "lot_2",
    "lot_3",
    "lot_4",
    "lot_5"
]

date_cols = [
    "transaction_date",
]

float_cols = [
    "property_value",
    "building_real_surface",
    "land_surface",
    "lot_1_surface",
    "lot_2_surface",
    "lot_3_surface",
    "lot_4_surface",
    "lot_5_surface"
]

int_cols = [
    "lots_count",
    "main_rooms_count",
]


def convert_chunk_types(chunk: pd.DataFrame) -> pd.DataFrame:
    # String / categorical columns
    for col in string_cols:
        if col in chunk.columns:
            chunk[col] = chunk[col].astype("string").str.strip()

    # Date columns
    for col in date_cols:
        if col in chunk.columns:
            chunk[col] = pd.to_datetime(
                chunk[col],
                errors="coerce"
            )

    # Float columns
    for col in float_cols:
        if col in chunk.columns:
            if chunk[col].dtype == "object" or str(chunk[col].dtype) == "string":
                chunk[col] = (
                    chunk[col]
                    .astype("string")
                    .str.replace(",", ".", regex=False)
                    .str.strip()
                )
            chunk[col] = pd.to_numeric(chunk[col], errors="coerce").astype("float64")

    # Integer columns
    for col in int_cols:
        if col in chunk.columns:
            if chunk[col].dtype == "object" or str(chunk[col].dtype) == "string":
                chunk[col] = chunk[col].astype("string").str.strip()
            chunk[col] = pd.to_numeric(chunk[col], errors="coerce").astype("Int64")

    return chunk


def convert_file_types_chunked(
    input_file: Path,
    output_file: Path,
    chunk_size: int = 50_000
) -> None:
    output_file.parent.mkdir(parents=True, exist_ok=True)

    first_write = True
    total_rows = 0

    for chunk_idx, chunk in enumerate(
        pd.read_csv(input_file, chunksize=chunk_size, dtype=str),
        start=1
    ):
        chunk = convert_chunk_types(chunk)

        chunk.to_csv(
            output_file,
            mode="w" if first_write else "a",
            index=False,
            header=first_write,
            encoding="utf-8"
        )

        first_write = False
        total_rows += len(chunk)

        print(f"Chunk {chunk_idx} written -> {len(chunk):,} rows | total: {total_rows:,}")

    print("\n✅ Done! Converted column data types")
    print(f"Saved file: {output_file}")
    print(f"Total rows written: {total_rows:,}")


convert_file_types_chunked(INPUT_FILE, OUTPUT_FILE, CHUNK_SIZE)

Chunk 1 written -> 500,000 rows | total: 500,000
Chunk 2 written -> 500,000 rows | total: 1,000,000
Chunk 3 written -> 500,000 rows | total: 1,500,000
Chunk 4 written -> 500,000 rows | total: 2,000,000
Chunk 5 written -> 500,000 rows | total: 2,500,000
Chunk 6 written -> 500,000 rows | total: 3,000,000
Chunk 7 written -> 500,000 rows | total: 3,500,000
Chunk 8 written -> 500,000 rows | total: 4,000,000
Chunk 9 written -> 500,000 rows | total: 4,500,000
Chunk 10 written -> 500,000 rows | total: 5,000,000
Chunk 11 written -> 500,000 rows | total: 5,500,000
Chunk 12 written -> 500,000 rows | total: 6,000,000
Chunk 13 written -> 500,000 rows | total: 6,500,000
Chunk 14 written -> 500,000 rows | total: 7,000,000
Chunk 15 written -> 500,000 rows | total: 7,500,000
Chunk 16 written -> 500,000 rows | total: 8,000,000
Chunk 17 written -> 500,000 rows | total: 8,500,000
Chunk 18 written -> 500,000 rows | total: 9,000,000
Chunk 19 written -> 500,000 rows | total: 9,500,000
Chunk 20 written -> 500

### **DATASET TABLE V3 (Features & Target + Categorical vs Numeric)**

| #  | Original Column Name           | Standardized            | Description                                       | Data Type      | Type                          |
| -- | ------------------------------ | ----------------------- | ------------------------------------------------- | -------------- | ----------------------------- |
| 0  | **No disposition**             | `transaction_id`        | Unique identifier of the transaction              | string         | Categorical / identifier      |
| 1  | **Date mutation**              | `transaction_date`      | Date when the property was sold                   | datetime64[ns] | Temporal                      |
| 2  | **Nature mutation**            | `transaction_type`      | Nature of transaction (sale, exchange, etc.)      | string         | Categorical                   |
| 3  | **Valeur fonciere**            | `property_value`        | Transaction price in euros                        | Float64        | Numeric (**TARGET**)          |
| 4  | **No voie**                    | `street_number`         | Street number of the property                     | string         | Categorical / identifier      |
| 5  | **B/T/Q**                      | `btq_code`              | Building / Type / Quarter code                    | string         | Categorical                   |
| 6  | **Type de voie**               | `street_type`           | Type of street (Rue, Avenue, etc.)                | string         | Categorical                   |
| 7  | **Code voie**                  | `street_id`             | Street identifier code                            | string         | Categorical / identifier      |
| 8  | **Voie**                       | `street_name`           | Full street name                                  | string         | Categorical                   |
| 9  | **Code postal**                | `postal_code`           | Postal code                                       | string         | Categorical / geographic code |
| 10 | **Commune**                    | `com_name`              | Commune name                                      | string         | Categorical                   |
| 11 | **Code departement**           | `dep_code`              | Department code                                   | string         | Categorical / geographic code |
| 12 | **Code commune**               | `com_code`              | Commune code                                      | string         | Categorical / geographic code |
| 13 | **Prefixe de section**         | `section_prefix`        | Section prefix used in cadastral identification   | string         | Categorical / cadastral code  |
| 14 | **Section**                    | `section`               | Cadastral section identifier                      | string         | Categorical / cadastral code  |
| 15 | **No plan**                    | `plot_number`           | Parcel / plot number                              | string         | Categorical / identifier      |
| 16 | **No Volume**                  | `volume_number`         | Volume number for special cadastral cases         | string         | Categorical / identifier      |
| 17 | **1er lot**                    | `lot_1`                 | Identifier of the first lot                       | string         | Categorical / identifier      |
| 18 | **Surface Carrez du 1er lot**  | `lot_1_surface`         | Carrez surface of the first lot in square meters  | Float64        | Numeric                       |
| 19 | **2eme lot**                   | `lot_2`                 | Identifier of the second lot                      | string         | Categorical / identifier      |
| 20 | **Surface Carrez du 2eme lot** | `lot_2_surface`         | Carrez surface of the second lot in square meters | Float64        | Numeric                       |
| 21 | **3eme lot**                   | `lot_3`                 | Identifier of the third lot                       | string         | Categorical / identifier      |
| 22 | **Surface Carrez du 3eme lot** | `lot_3_surface`         | Carrez surface of the third lot in square meters  | Float64        | Numeric                       |
| 23 | **4eme lot**                   | `lot_4`                 | Identifier of the fourth lot                      | string         | Categorical / identifier      |
| 24 | **Surface Carrez du 4eme lot** | `lot_4_surface`         | Carrez surface of the fourth lot in square meters | Float64        | Numeric                       |
| 25 | **5eme lot**                   | `lot_5`                 | Identifier of the fifth lot                       | string         | Categorical / identifier      |
| 26 | **Surface Carrez du 5eme lot** | `lot_5_surface`         | Carrez surface of the fifth lot in square meters  | Float64        | Numeric                       |
| 27 | **Nombre de lots**             | `lots_count`            | Number of lots in the transaction                 | Int64          | Numeric count                 |
| 28 | **Code type local**            | `property_type_code`    | Code for property type                            | string         | Categorical code              |
| 29 | **Type local**                 | `property_type`         | Property type (Apartment, House, etc.)            | string         | Categorical                   |
| 30 | **Surface reelle bati**        | `building_real_surface` | Built area in square meters                       | Float64        | Numeric                       |
| 31 | **Nombre pieces principales**  | `main_rooms_count`      | Number of main rooms                              | Int64          | Numeric count                 |
| 32 | **Nature culture**             | `land_nature`           | General type of land usage                        | string         | Categorical                   |
| 33 | **Nature culture speciale**    | `land_nature_special`   | Specific type of land usage                       | string         | Categorical                   |
| 34 | **Surface terrain**            | `land_surface`          | Land area in square meters                        | Float64        | Numeric                       |
| 35 | *(derived)*                    | `surface_type`          | Type of surface: building / land / combined       | string         | Derived categorical           |


<hr>

## 4️⃣ NULL VALUES


<style>
h1 {
    text-align: center;
    color: black;
    font-weight: bold;
}
</style>

<style>
h2 {
    text-align: left;
    color: black;
    font-weight: bold;
}
</style>

<style>
h3 {
    text-align: left;
    color: black;
    font-weight: bold;
}
</style>

<style>
h4 {
    text-align: left;
    color: black;
    font-weight: bold;
}
</style>
<hr>

→ WHY? Missing data can bias results.

### **INFO - NULLS COUNT (BEFORE)**

In [15]:
import pandas as pd

FILE = "../data/processed/TYPES_ValeursFoncieres.csv"
CHUNK_SIZE = 500_000
dtype_map = {
    # categorical / identifiers
    "transaction_id": "string",
    "transaction_date": "string",  # convert to datetime after loading
    "transaction_type": "string",
    "street_number": "string",
    "btq_code": "string",
    "street_type": "string",
    "street_id": "string",
    "street_name": "string",
    "postal_code": "string",
    "com_name": "string",
    "dep_code": "string",
    "com_code": "string",
    "section_prefix": "string",
    "section": "string",
    "plot_number": "string",
    "volume_number": "string",
    "property_type_code": "string",  # keep as string (categorical)
    "property_type": "string",
    "land_nature": "string",
    "land_nature_special": "string",
    "surface_type": "string",

    # categorical lot identifiers
    "lot_1": "string",
    "lot_2": "string",
    "lot_3": "string",
    "lot_4": "string",
    "lot_5": "string",

    # numeric float (use nullable Float64 for consistency)
    "property_value": "Float64",
    "building_real_surface": "Float64",
    "land_surface": "Float64",
    "lot_1_surface": "Float64",
    "lot_2_surface": "Float64",
    "lot_3_surface": "Float64",
    "lot_4_surface": "Float64",
    "lot_5_surface": "Float64",

    # numeric integer (nullable)
    "lots_count": "Int64",
    "main_rooms_count": "Int64",
}


total_rows = 0
columns = None
non_null_counts = None
dtypes = None

for chunk in pd.read_csv(
    FILE,
    chunksize=CHUNK_SIZE,
    dtype=dtype_map,
    #parse_dates=["transaction_date"],
    #dayfirst=True,        # IMPORTANT! date format: dd/mm/yyyy
    low_memory=False      # removes DtypeWarning
):
    if columns is None:
        columns = chunk.columns.tolist()
        non_null_counts = {col: 0 for col in columns}
        dtypes = chunk.dtypes.astype(str).to_dict()

    total_rows += len(chunk)

    for col in columns:
        non_null_counts[col] += chunk[col].notna().sum()

info_df = pd.DataFrame({
    "column": columns,
    "non_null_count": [non_null_counts[col] for col in columns],
    "null_count": [total_rows - non_null_counts[col] for col in columns],
    "dtype": [dtypes[col] for col in columns]
})

print(f"RangeIndex: {total_rows} entries, 0 to {total_rows - 1}")
print(f"Data columns (total {len(columns)} columns):")
display(info_df)
print(f"dtypes: {info_df['dtype'].value_counts().to_dict()}")
print("memory usage: not computed (chunked mode)")


RangeIndex: 20102739 entries, 0 to 20102738
Data columns (total 36 columns):


,column,non_null_count,null_count,dtype
0,transaction_id,20102739,0,string
1,transaction_date,20102739,0,string
2,transaction_type,20102739,0,string
3,property_value,19908349,194390,Float64
4,street_number,12628228,7474511,string
5,btq_code,904998,19197741,string
6,street_type,12160680,7942059,string
7,street_id,19952257,150482,string
8,street_name,19951476,151263,string
9,postal_code,19951139,151600,string


dtypes: {'string': 26, 'Float64': 8, 'Int64': 2}
memory usage: not computed (chunked mode)


### **DEAL with NULLS FINAL**

In [19]:
from pathlib import Path
import pandas as pd

INPUT_FILE = Path("../data/processed/TYPES_ValeursFoncieres.csv")
OUTPUT_FILE = Path("../data/processed/NULLS_ValeursFoncieres.csv")

CHUNK_SIZE = 500_000

dtype_map = {
    # categorical / identifiers
    "transaction_id": "string",
    "transaction_date": "string",
    "transaction_type": "string",
    "street_number": "string",
    "btq_code": "string",
    "street_type": "string",
    "street_id": "string",
    "street_name": "string",
    "postal_code": "string",
    "com_name": "string",
    "dep_code": "string",
    "com_code": "string",
    "section_prefix": "string",
    "section": "string",
    "plot_number": "string",
    "volume_number": "string",
    "property_type_code": "string",
    "property_type": "string",
    "land_nature": "string",
    "land_nature_special": "string",
    "surface_type": "string",

    # categorical lot identifiers
    "lot_1": "string",
    "lot_2": "string",
    "lot_3": "string",
    "lot_4": "string",
    "lot_5": "string",

    # numeric float
    "property_value": "Float64",
    "building_real_surface": "Float64",
    "land_surface": "Float64",
    "lot_1_surface": "Float64",
    "lot_2_surface": "Float64",
    "lot_3_surface": "Float64",
    "lot_4_surface": "Float64",
    "lot_5_surface": "Float64",

    # numeric integer
    "lots_count": "Int64",
    "main_rooms_count": "Int64",
}


def handle_nulls(chunk: pd.DataFrame) -> pd.DataFrame:

    # --------------------------------------------------------------------------------
    # Column: property_value (TARGET)
    # --------------------------------------------------------------------------------
    # target cannot be imputed safely
    chunk = chunk.dropna(subset=["property_value"]).copy()

    # --------------------------------------------------------------------------------
    # Column: building_real_surface
    # --------------------------------------------------------------------------------
    chunk["building_real_surface"] = chunk["building_real_surface"].fillna(0.0)

    # --------------------------------------------------------------------------------
    # Columns: lot_1_surface, lot_2_surface, lot_3_surface, lot_4_surface, lot_5_surface
    # --------------------------------------------------------------------------------
    chunk["lot_1_surface"] = chunk["lot_1_surface"].fillna(0.0)
    chunk["lot_2_surface"] = chunk["lot_2_surface"].fillna(0.0)
    chunk["lot_3_surface"] = chunk["lot_3_surface"].fillna(0.0)
    chunk["lot_4_surface"] = chunk["lot_4_surface"].fillna(0.0)
    chunk["lot_5_surface"] = chunk["lot_5_surface"].fillna(0.0)

    # --------------------------------------------------------------------------------
    # Column: main_rooms_count
    # --------------------------------------------------------------------------------
    # if main_rooms_count is missing and building_real_surface exists, fill with 0
    mask = (
        chunk["main_rooms_count"].isna() &
        (chunk["building_real_surface"] >= 0)
    )
    chunk.loc[mask, "main_rooms_count"] = 0

    # --------------------------------------------------------------------------------
    # Column: land_surface
    # --------------------------------------------------------------------------------
    # if land_surface is missing and building_real_surface exists, fill with 0
    mask = (
        chunk["land_surface"].isna() &
        (chunk["building_real_surface"] > 0)
    )
    chunk.loc[mask, "land_surface"] = 0.0

    # --------------------------------------------------------------------------------
    # Column: land_surface
    # --------------------------------------------------------------------------------
    chunk["land_surface"] = chunk["land_surface"].fillna(0.0)

    # --------------------------------------------------------------------------------
    # Column: lots_count
    # --------------------------------------------------------------------------------
    chunk["lots_count"] = chunk["lots_count"].fillna(0)

    # --------------------------------------------------------------------------------
    # Column: main_rooms_count
    # --------------------------------------------------------------------------------
    chunk["main_rooms_count"] = chunk["main_rooms_count"].fillna(0)

    # --------------------------------------------------------------------------------
    # Column: transaction_id
    # --------------------------------------------------------------------------------
    chunk["transaction_id"] = chunk["transaction_id"].fillna("Unknown")

    # --------------------------------------------------------------------------------
    # Column: transaction_date
    # --------------------------------------------------------------------------------
    chunk["transaction_date"] = chunk["transaction_date"].fillna("Unknown")

    # --------------------------------------------------------------------------------
    # Column: transaction_type
    # --------------------------------------------------------------------------------
    chunk["transaction_type"] = chunk["transaction_type"].fillna("Unknown")

    # --------------------------------------------------------------------------------
    # Column: street_number
    # --------------------------------------------------------------------------------
    chunk["street_number"] = chunk["street_number"].fillna("Unknown")

    # --------------------------------------------------------------------------------
    # Column: btq_code
    # --------------------------------------------------------------------------------
    chunk["btq_code"] = chunk["btq_code"].fillna("Unknown")

    # --------------------------------------------------------------------------------
    # Column: street_type
    # --------------------------------------------------------------------------------
    chunk["street_type"] = chunk["street_type"].fillna("Unknown")

    # --------------------------------------------------------------------------------
    # Column: street_id
    # --------------------------------------------------------------------------------
    chunk["street_id"] = chunk["street_id"].fillna("Unknown")

    # --------------------------------------------------------------------------------
    # Column: street_name
    # --------------------------------------------------------------------------------
    chunk["street_name"] = chunk["street_name"].fillna("Unknown")

    # --------------------------------------------------------------------------------
    # Column: postal_code
    # --------------------------------------------------------------------------------
    chunk["postal_code"] = chunk["postal_code"].fillna("Unknown")

    # --------------------------------------------------------------------------------
    # Column: com_name
    # --------------------------------------------------------------------------------
    chunk["com_name"] = chunk["com_name"].fillna("Unknown")

    # --------------------------------------------------------------------------------
    # Column: dep_code
    # --------------------------------------------------------------------------------
    chunk["dep_code"] = chunk["dep_code"].fillna("Unknown")

    # --------------------------------------------------------------------------------
    # Column: com_code
    # --------------------------------------------------------------------------------
    chunk["com_code"] = chunk["com_code"].fillna("Unknown")

    # --------------------------------------------------------------------------------
    # Column: section_prefix
    # --------------------------------------------------------------------------------
    chunk["section_prefix"] = chunk["section_prefix"].fillna("Unknown")

    # --------------------------------------------------------------------------------
    # Column: section
    # --------------------------------------------------------------------------------
    chunk["section"] = chunk["section"].fillna("Unknown")

    # --------------------------------------------------------------------------------
    # Column: plot_number
    # --------------------------------------------------------------------------------
    chunk["plot_number"] = chunk["plot_number"].fillna("Unknown")

    # --------------------------------------------------------------------------------
    # Column: volume_number
    # --------------------------------------------------------------------------------
    chunk["volume_number"] = chunk["volume_number"].fillna("Unknown")

    # --------------------------------------------------------------------------------
    # Columns: lot_1, lot_2, lot_3, lot_4, lot_5
    # --------------------------------------------------------------------------------
    chunk["lot_1"] = chunk["lot_1"].fillna("Unknown")
    chunk["lot_2"] = chunk["lot_2"].fillna("Unknown")
    chunk["lot_3"] = chunk["lot_3"].fillna("Unknown")
    chunk["lot_4"] = chunk["lot_4"].fillna("Unknown")
    chunk["lot_5"] = chunk["lot_5"].fillna("Unknown")

    # --------------------------------------------------------------------------------
    # Column: property_type_code
    # --------------------------------------------------------------------------------
    chunk["property_type_code"] = chunk["property_type_code"].fillna("Unknown")

    # --------------------------------------------------------------------------------
    # Column: property_type
    # --------------------------------------------------------------------------------
    chunk["property_type"] = chunk["property_type"].fillna("Unknown")

    # --------------------------------------------------------------------------------
    # Column: land_nature
    # --------------------------------------------------------------------------------
    chunk["land_nature"] = chunk["land_nature"].fillna("Unknown")

    # --------------------------------------------------------------------------------
    # Column: land_nature_special
    # --------------------------------------------------------------------------------
    chunk["land_nature_special"] = chunk["land_nature_special"].fillna("Unknown")

    # --------------------------------------------------------------------------------
    # Column: surface_type
    # --------------------------------------------------------------------------------
    chunk["surface_type"] = chunk["surface_type"].fillna("Unknown")

    # --------------------------------------------------------------------------------
    # Restore numeric dtypes
    # --------------------------------------------------------------------------------
    chunk["property_value"] = pd.to_numeric(chunk["property_value"], errors="coerce")
    chunk["building_real_surface"] = pd.to_numeric(chunk["building_real_surface"], errors="coerce")
    chunk["land_surface"] = pd.to_numeric(chunk["land_surface"], errors="coerce")
    chunk["lot_1_surface"] = pd.to_numeric(chunk["lot_1_surface"], errors="coerce")
    chunk["lot_2_surface"] = pd.to_numeric(chunk["lot_2_surface"], errors="coerce")
    chunk["lot_3_surface"] = pd.to_numeric(chunk["lot_3_surface"], errors="coerce")
    chunk["lot_4_surface"] = pd.to_numeric(chunk["lot_4_surface"], errors="coerce")
    chunk["lot_5_surface"] = pd.to_numeric(chunk["lot_5_surface"], errors="coerce")

    chunk["lots_count"] = pd.to_numeric(chunk["lots_count"], errors="coerce").astype("Int64")
    chunk["main_rooms_count"] = pd.to_numeric(chunk["main_rooms_count"], errors="coerce").astype("Int64")

    return chunk


def process_nulls_chunked(input_file: Path, output_file: Path, chunk_size: int = 500_000) -> None:
    output_file.parent.mkdir(parents=True, exist_ok=True)

    first_write = True
    total_rows_read = 0
    total_rows_written = 0
    total_rows_dropped = 0

    for i, chunk in enumerate(
        pd.read_csv(
            input_file,
            chunksize=chunk_size,
            dtype=dtype_map,
            low_memory=False
        ),
        start=1
    ):
        rows_before = len(chunk)
        total_rows_read += rows_before

        chunk = handle_nulls(chunk)

        rows_after = len(chunk)
        total_rows_written += rows_after
        total_rows_dropped += rows_before - rows_after

        chunk.to_csv(
            output_file,
            mode="w" if first_write else "a",
            index=False,
            header=first_write,
            encoding="utf-8"
        )

        first_write = False

        print(
            f"Chunk {i} processed -> "
            f"read: {rows_before:,} | kept: {rows_after:,} | "
            f"dropped: {rows_before - rows_after:,} | total kept: {total_rows_written:,}"
        )

    print("\n✅ Done! Null values handled")
    print(f"Saved file: {output_file}")
    print(f"Total rows read: {total_rows_read:,}")
    print(f"Total rows dropped: {total_rows_dropped:,}")
    print(f"Total rows written: {total_rows_written:,}")


process_nulls_chunked(INPUT_FILE, OUTPUT_FILE, CHUNK_SIZE)

Chunk 1 processed -> read: 500,000 | kept: 496,656 | dropped: 3,344 | total kept: 496,656
Chunk 2 processed -> read: 500,000 | kept: 496,789 | dropped: 3,211 | total kept: 993,445
Chunk 3 processed -> read: 500,000 | kept: 485,317 | dropped: 14,683 | total kept: 1,478,762
Chunk 4 processed -> read: 500,000 | kept: 487,634 | dropped: 12,366 | total kept: 1,966,396
Chunk 5 processed -> read: 500,000 | kept: 497,421 | dropped: 2,579 | total kept: 2,463,817
Chunk 6 processed -> read: 500,000 | kept: 497,411 | dropped: 2,589 | total kept: 2,961,228
Chunk 7 processed -> read: 500,000 | kept: 484,285 | dropped: 15,715 | total kept: 3,445,513
Chunk 8 processed -> read: 500,000 | kept: 498,051 | dropped: 1,949 | total kept: 3,943,564
Chunk 9 processed -> read: 500,000 | kept: 496,009 | dropped: 3,991 | total kept: 4,439,573
Chunk 10 processed -> read: 500,000 | kept: 496,176 | dropped: 3,824 | total kept: 4,935,749
Chunk 11 processed -> read: 500,000 | kept: 495,767 | dropped: 4,233 | total kep

### **INFO - NULLS COUNT (AFTER)**

In [20]:
import pandas as pd

print("Re-checking data types and null counts after null handling...")

FILE = "../data/processed/NULLS_ValeursFoncieres.csv"
CHUNK_SIZE = 500_000

dtype_map = {
    # categorical / identifiers
    "transaction_id": "string",
    "transaction_date": "string",  # convert to datetime after loading
    "transaction_type": "string",
    "street_number": "string",
    "btq_code": "string",
    "street_type": "string",
    "street_id": "string",
    "street_name": "string",
    "postal_code": "string",
    "com_name": "string",
    "dep_code": "string",
    "com_code": "string",
    "section_prefix": "string",
    "section": "string",
    "plot_number": "string",
    "volume_number": "string",
    "property_type_code": "string",  # keep as string (categorical)
    "property_type": "string",
    "land_nature": "string",
    "land_nature_special": "string",
    "surface_type": "string",

    # lot identifiers
    "lot_1": "string",
    "lot_2": "string",
    "lot_3": "string",
    "lot_4": "string",
    "lot_5": "string",

    # numeric (use nullable Float64 for consistency)
    "property_value": "Float64",
    "building_real_surface": "Float64",
    "land_surface": "Float64",
    "lot_1_surface": "Float64",
    "lot_2_surface": "Float64",
    "lot_3_surface": "Float64",
    "lot_4_surface": "Float64",
    "lot_5_surface": "Float64",

    # integer (nullable)
    "lots_count": "Int64",
    "main_rooms_count": "Int64",
}


total_rows = 0
columns = None
non_null_counts = None
dtypes = None

for chunk in pd.read_csv(
    FILE,
    chunksize=CHUNK_SIZE,
    dtype=dtype_map,
    #parse_dates=["transaction_date"],
    #dayfirst=True,        # IMPORTANT! date format: dd/mm/yyyy
    low_memory=False      # removes DtypeWarning
):
    if columns is None:
        columns = chunk.columns.tolist()
        non_null_counts = {col: 0 for col in columns}
        dtypes = chunk.dtypes.astype(str).to_dict()

    total_rows += len(chunk)

    for col in columns:
        non_null_counts[col] += chunk[col].notna().sum()

info_df = pd.DataFrame({
    "column": columns,
    "non_null_count": [non_null_counts[col] for col in columns],
    "null_count": [total_rows - non_null_counts[col] for col in columns],
    "dtype": [dtypes[col] for col in columns]
})

print(f"RangeIndex: {total_rows} entries, 0 to {total_rows - 1}")
print(f"Data columns (total {len(columns)} columns):")
display(info_df)
print(f"dtypes: {info_df['dtype'].value_counts().to_dict()}")
print("memory usage: not computed (chunked mode)")


Re-checking data types and null counts after null handling...
RangeIndex: 19908349 entries, 0 to 19908348
Data columns (total 36 columns):


,column,non_null_count,null_count,dtype
0,transaction_id,19908349,0,string
1,transaction_date,19908349,0,string
2,transaction_type,19908349,0,string
3,property_value,19908349,0,Float64
4,street_number,19908349,0,string
5,btq_code,19908349,0,string
6,street_type,19908349,0,string
7,street_id,19908349,0,string
8,street_name,19908349,0,string
9,postal_code,19908349,0,string


dtypes: {'string': 26, 'Float64': 8, 'Int64': 2}
memory usage: not computed (chunked mode)


<hr>

## 5️⃣ DUPLICATES


<style>
h1 {
    text-align: center;
    color: black;
    font-weight: bold;
}
</style>

<style>
h2 {
    text-align: left;
    color: black;
    font-weight: bold;
}
</style>

<style>
h3 {
    text-align: left;
    color: black;
    font-weight: bold;
}
</style>

<style>
h4 {
    text-align: left;
    color: black;
    font-weight: bold;
}
</style>
<hr>

### **SHAPE & HEAD - BEFORE Dealing with Duplicates**

In [21]:
import pandas as pd

FILE = "../data/processed/NULLS_ValeursFoncieres.csv"
CHUNK_SIZE = 500_000

dtype_map = {
    # categorical / identifiers
    "transaction_id": "string",
    "transaction_date": "string",  # convert to datetime after loading
    "transaction_type": "string",
    "street_number": "string",
    "btq_code": "string",
    "street_type": "string",
    "street_id": "string",
    "street_name": "string",
    "postal_code": "string",
    "com_name": "string",
    "dep_code": "string",
    "com_code": "string",
    "section_prefix": "string",
    "section": "string",
    "plot_number": "string",
    "volume_number": "string",
    "property_type_code": "string",  # keep as string (categorical)
    "property_type": "string",
    "land_nature": "string",
    "land_nature_special": "string",
    "surface_type": "string",

    # lot identifiers
    "lot_1": "string",
    "lot_2": "string",
    "lot_3": "string",
    "lot_4": "string",
    "lot_5": "string",

    # numeric (use nullable Float64 for consistency)
    "property_value": "Float64",
    "building_real_surface": "Float64",
    "land_surface": "Float64",
    "lot_1_surface": "Float64",
    "lot_2_surface": "Float64",
    "lot_3_surface": "Float64",
    "lot_4_surface": "Float64",
    "lot_5_surface": "Float64",

    # integer (nullable)
    "lots_count": "Int64",
    "main_rooms_count": "Int64",
}

# ---------------------------
# Get full dataset shape
# ---------------------------
total_rows = 0
columns = None

for chunk in pd.read_csv(
    FILE,
    chunksize=CHUNK_SIZE,
    dtype=dtype_map,
    low_memory=False
):
    total_rows += len(chunk)
    if columns is None:
        columns = chunk.columns

print("Shape:", total_rows, "rows and", len(columns), "columns")

# ---------------------------
# Display head with correct dtypes
# ---------------------------
df_head = pd.read_csv(
    FILE,
    nrows=5,
    dtype=dtype_map,
    low_memory=False
)

display(df_head)

Shape: 19908349 rows and 36 columns


,transaction_id,transaction_date,transaction_type,property_value,street_number,btq_code,street_type,street_id,street_name,postal_code,com_name,dep_code,com_code,section_prefix,section,plot_number,volume_number,lot_1,lot_1_surface,lot_2,lot_2_surface,lot_3,lot_3_surface,lot_4,lot_4_surface,lot_5,lot_5_surface,lots_count,property_type_code,property_type,building_real_surface,main_rooms_count,land_nature,land_nature_special,land_surface,surface_type
0,000001,2020-07-01,Sale,31234.16,Unknown,Unknown,Unknown,B064,SAINT JULIEN,1560,SAINT-JULIEN-SUR-REYSSOUZE,01,367,Unknown,A,8,Unknown,Unknown,0.0,Unknown,0.0,Unknown,0.0,Unknown,0.0,Unknown,0.0,0,Unknown,Unknown,0.0,0,AB,Unknown,1192.0,land
1,000001,2020-07-01,Sale,278000.0,Unknown,Unknown,Unknown,B188,A LA PEROUSE,1250,CORVEISSIAT,01,125,Unknown,C,509,Unknown,Unknown,0.0,Unknown,0.0,Unknown,0.0,Unknown,0.0,Unknown,0.0,0,Unknown,Unknown,0.0,0,BS,Unknown,10092.0,land
2,000001,2020-07-01,Sale,278000.0,Unknown,Unknown,Unknown,B188,A LA PEROUSE,1250,CORVEISSIAT,01,125,Unknown,C,510,Unknown,Unknown,0.0,Unknown,0.0,Unknown,0.0,Unknown,0.0,Unknown,0.0,0,Unknown,Unknown,0.0,0,L,Unknown,4570.0,land
3,000001,2020-07-01,Sale,278000.0,Unknown,Unknown,Unknown,B079,AUX COMMUNS,1250,CORVEISSIAT,01,125,Unknown,ZL,96,Unknown,Unknown,0.0,Unknown,0.0,Unknown,0.0,Unknown,0.0,Unknown,0.0,0,Unknown,Unknown,0.0,0,BS,Unknown,5750.0,land
4,000001,2020-07-01,Sale,278000.0,Unknown,Unknown,Unknown,B033,EN COMBARNAUD,1250,SIMANDRE-SUR-SURAN,01,408,Unknown,A,14,Unknown,Unknown,0.0,Unknown,0.0,Unknown,0.0,Unknown,0.0,Unknown,0.0,0,Unknown,Unknown,0.0,0,BT,Unknown,648170.0,land


### **DUPLICATES STATS (percentage and count of duplicates)**

In [ ]:
import pandas as pd
from pathlib import Path
import hashlib

INPUT_FILE = Path("../data/processed/NULLS_ValeursFoncieres.csv")

chunk_size = 1_000_000

dtype_map = {
    "transaction_id": "string",
    "transaction_date": "string",
    "transaction_type": "string",
    "street_number": "string",
    "btq_code": "string",
    "street_type": "string",
    "street_id": "string",
    "street_name": "string",
    "postal_code": "string",
    "com_name": "string",
    "dep_code": "string",
    "com_code": "string",
    "section_prefix": "string",
    "section": "string",
    "plot_number": "string",
    "volume_number": "string",
    "property_type_code": "string",
    "property_type": "string",
    "land_nature": "string",
    "land_nature_special": "string",
    "surface_type": "string",

    "lot_1": "string",
    "lot_2": "string",
    "lot_3": "string",
    "lot_4": "string",
    "lot_5": "string",

    "property_value": "Float64",
    "building_real_surface": "Float64",
    "land_surface": "Float64",
    "lot_1_surface": "Float64",
    "lot_2_surface": "Float64",
    "lot_3_surface": "Float64",
    "lot_4_surface": "Float64",
    "lot_5_surface": "Float64",

    "lots_count": "Int64",
    "main_rooms_count": "Int64",
}

total_rows = 0
total_duplicates = 0
seen_hashes = set()


def hash_row(row):
    # No cleaning needed, just stringify
    row_string = "|".join(map(str, row))
    return hashlib.md5(row_string.encode("utf-8")).hexdigest()


for chunk in pd.read_csv(
    INPUT_FILE,
    chunksize=chunk_size,
    dtype=dtype_map,
    low_memory=False
):
    total_rows += len(chunk)

    for row in chunk.itertuples(index=False, name=None):
        h = hash_row(row)

        if h in seen_hashes:
            total_duplicates += 1
        else:
            seen_hashes.add(h)


unique_rows = total_rows - total_duplicates
duplicate_percentage = (total_duplicates / total_rows) * 100

print(f"Total rows (including duplicates): {total_rows:,}")
print(f"Duplicate rows (excluding first occurrences): {total_duplicates:,}")
print(f"Duplicate percentage: {duplicate_percentage:.2f}%")
print(f"Total rows without duplicates (including first occurrences): {unique_rows:,}")

Total rows (including duplicates): 19,908,349
Duplicate rows (excluding first occurrences): 1,388,539
Duplicate percentage: 6.97%
Total rows without duplicates (including first occurrences): 18,519,810


### **DUPLICATES ROWS & NULLS**

In [23]:
from pathlib import Path
import pandas as pd
import hashlib

FILE = Path("../data/processed/NULLS_ValeursFoncieres.csv")
CHUNK_SIZE = 500_000


dtype_map = {
    # categorical / identifiers
    "transaction_id": "string",
    "transaction_date": "string",  # convert to datetime after loading
    "transaction_type": "string",
    "street_number": "string",
    "btq_code": "string",
    "street_type": "string",
    "street_id": "string",
    "street_name": "string",
    "postal_code": "string",
    "com_name": "string",
    "dep_code": "string",
    "com_code": "string",
    "section_prefix": "string",
    "section": "string",
    "plot_number": "string",
    "volume_number": "string",
    "property_type_code": "string",  # keep as string (categorical)
    "property_type": "string",
    "land_nature": "string",
    "land_nature_special": "string",
    "surface_type": "string",

    # lot identifiers
    "lot_1": "string",
    "lot_2": "string",
    "lot_3": "string",
    "lot_4": "string",
    "lot_5": "string",

    # numeric (use nullable Float64 for consistency)
    "property_value": "Float64",
    "building_real_surface": "Float64",
    "land_surface": "Float64",
    "lot_1_surface": "Float64",
    "lot_2_surface": "Float64",
    "lot_3_surface": "Float64",
    "lot_4_surface": "Float64",
    "lot_5_surface": "Float64",

    # integer (nullable)
    "lots_count": "Int64",
    "main_rooms_count": "Int64",
}

def row_hash(row: pd.Series) -> str:
    values = row.fillna("<NA>").astype(str).tolist()
    row_str = "||".join(values)
    return hashlib.md5(row_str.encode("utf-8")).hexdigest()

total_rows = 0
duplicate_rows_count = 0
seen_hashes = set()

null_counts = None
columns = None
duplicate_samples = []

for chunk in pd.read_csv(
    FILE,
    chunksize=CHUNK_SIZE,
    dtype=dtype_map,
    low_memory=False
):
    if columns is None:
        columns = chunk.columns.tolist()
        null_counts = {col: 0 for col in columns}

    total_rows += len(chunk)

    # Null counts
    for col in columns:
        null_counts[col] += chunk[col].isna().sum()

    # Duplicate detection across all chunks
    chunk_hashes = chunk.apply(row_hash, axis=1)

    is_duplicate = chunk_hashes.isin(seen_hashes)
    duplicate_rows_count += is_duplicate.sum()

    # Save a few duplicate row samples for display
    if is_duplicate.any() and len(duplicate_samples) < 20:
        sample_dups = chunk.loc[is_duplicate].head(20 - len(duplicate_samples))
        duplicate_samples.append(sample_dups)

    # Add only new hashes
    new_hashes = chunk_hashes.loc[~is_duplicate]
    seen_hashes.update(new_hashes.tolist())

# Build null percentage summary
null_pct = {
    col: round(null_counts[col] / total_rows * 100, 2)
    for col in columns
}

nulls_df = pd.DataFrame({
    "column": columns,
    "null_count": [null_counts[col] for col in columns],
    "null_pct": [null_pct[col] for col in columns]
}).sort_values("null_pct", ascending=False)


# Print results
print(100 * "-")
print(f"Number of duplicate rows: {duplicate_rows_count}")
print("Nulls percentage in each column:")
display(nulls_df)
print(f"Duplicate rows percentage: {duplicate_rows_count / total_rows * 100:.2f}%")
print(100 * "-")

# Display duplicate samples if any
if duplicate_rows_count > 0:
    print("\nDuplicate row samples:")
    display(pd.concat(duplicate_samples, ignore_index=True))
else:
    print("\nNo duplicate rows found.")

----------------------------------------------------------------------------------------------------
Number of duplicate rows: 608
Nulls percentage in each column:


,column,null_count,null_pct
0,transaction_id,0,0.0
1,transaction_date,0,0.0
2,transaction_type,0,0.0
3,property_value,0,0.0
4,street_number,0,0.0
5,btq_code,0,0.0
6,street_type,0,0.0
7,street_id,0,0.0
8,street_name,0,0.0
9,postal_code,0,0.0


Duplicate rows percentage: 0.00%
----------------------------------------------------------------------------------------------------

Duplicate row samples:


,transaction_id,transaction_date,transaction_type,property_value,street_number,btq_code,street_type,street_id,street_name,postal_code,com_name,dep_code,com_code,section_prefix,section,plot_number,volume_number,lot_1,lot_1_surface,lot_2,lot_2_surface,lot_3,lot_3_surface,lot_4,lot_4_surface,lot_5,lot_5_surface,lots_count,property_type_code,property_type,building_real_surface,main_rooms_count,land_nature,land_nature_special,land_surface,surface_type
0,000001,2020-10-20,Sale,5308941.0,36,Unknown,RLE,0070,DES ATTES,97436,SAINT-LEU,974,13,Unknown,AV,1712,Unknown,Unknown,0.0,Unknown,0.0,Unknown,0.0,Unknown,0.0,Unknown,0.0,0,2,Apartment,73.0,4,S,Unknown,35.0,combined
1,000001,2020-10-20,Sale,5308941.0,36,Unknown,RLE,0070,DES ATTES,97436,SAINT-LEU,974,13,Unknown,AV,1712,Unknown,Unknown,0.0,Unknown,0.0,Unknown,0.0,Unknown,0.0,Unknown,0.0,0,2,Apartment,34.0,2,S,Unknown,35.0,combined
2,000001,2020-10-20,Sale,5308941.0,36,Unknown,RLE,0070,DES ATTES,97436,SAINT-LEU,974,13,Unknown,AV,1712,Unknown,Unknown,0.0,Unknown,0.0,Unknown,0.0,Unknown,0.0,Unknown,0.0,0,2,Apartment,73.0,4,S,Unknown,35.0,combined
3,000001,2020-10-20,Sale,5308941.0,36,Unknown,RLE,0070,DES ATTES,97436,SAINT-LEU,974,13,Unknown,AV,1712,Unknown,Unknown,0.0,Unknown,0.0,Unknown,0.0,Unknown,0.0,Unknown,0.0,0,2,Apartment,73.0,4,S,Unknown,35.0,combined
4,000001,2020-10-20,Sale,5308941.0,36,Unknown,RLE,0070,DES ATTES,97436,SAINT-LEU,974,13,Unknown,AV,1712,Unknown,Unknown,0.0,Unknown,0.0,Unknown,0.0,Unknown,0.0,Unknown,0.0,0,2,Apartment,34.0,2,S,Unknown,35.0,combined
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
61,000001,2024-10-29,Sale in future state of completion,173500.0,Unknown,Unknown,Unknown,B235,LA LAGREMEUSE HAUTE,13290,AIX EN PROVENCE,13,1,Unknown,KV,214,Unknown,141,0.0,Unknown,0.0,Unknown,0.0,Unknown,0.0,Unknown,0.0,1,Unknown,Unknown,0.0,0,Unknown,Unknown,0.0,Unknown
62,000001,2024-11-14,Sale,164000.0,Unknown,Unknown,RUE,0400,JOSEPH LE MAT,29680,ROSCOFF,29,239,Unknown,AI,845,Unknown,Unknown,0.0,Unknown,0.0,Unknown,0.0,Unknown,0.0,Unknown,0.0,0,Unknown,Unknown,0.0,0,S,Unknown,35.0,land
63,000001,2024-03-25,Sale in future state of completion,207000.0,22,Unknown,AV,0380,LOUIS PASTEUR,84130,LE PONTET,84,92,Unknown,AL,14,Unknown,31,0.0,Unknown,0.0,Unknown,0.0,Unknown,0.0,Unknown,0.0,1,2,Apartment,79.0,4,Unknown,Unknown,0.0,building
64,000001,2024-03-25,Sale in future state of completion,207000.0,22,Unknown,AV,0380,LOUIS PASTEUR,84130,LE PONTET,84,92,Unknown,AL,14,Unknown,68,0.0,Unknown,0.0,Unknown,0.0,Unknown,0.0,Unknown,0.0,1,3,Outbuilding,0.0,0,Unknown,Unknown,0.0,Unknown


### **DEAL with DUPLICATES**

In [25]:
from pathlib import Path
import pandas as pd

INPUT_FILE = Path("../data/processed/NULLS_ValeursFoncieres.csv")
OUTPUT_FILE = Path("../data/processed/DUPLICATES_ValeursFoncieres.csv")

CHUNK_SIZE = 500_000

dtype_map = {
    "transaction_id": "string",
    "transaction_date": "string",
    "transaction_type": "string",
    "street_number": "string",
    "btq_code": "string",
    "street_type": "string",
    "street_id": "string",
    "street_name": "string",
    "postal_code": "string",
    "com_name": "string",
    "dep_code": "string",
    "com_code": "string",
    "section_prefix": "string",
    "section": "string",
    "plot_number": "string",
    "volume_number": "string",
    "property_type_code": "string",
    "property_type": "string",
    "land_nature": "string",
    "land_nature_special": "string",
    "surface_type": "string",

    "lot_1": "string",
    "lot_2": "string",
    "lot_3": "string",
    "lot_4": "string",
    "lot_5": "string",

    "property_value": "Float64",
    "building_real_surface": "Float64",
    "land_surface": "Float64",
    "lot_1_surface": "Float64",
    "lot_2_surface": "Float64",
    "lot_3_surface": "Float64",
    "lot_4_surface": "Float64",
    "lot_5_surface": "Float64",

    "lots_count": "Int64",
    "main_rooms_count": "Int64",
}


def remove_duplicates_fast(
    input_file: Path,
    output_file: Path,
    chunk_size: int = 500_000
) -> None:
    output_file.parent.mkdir(parents=True, exist_ok=True)

    seen_hashes = set()
    first_write = True

    total_read = 0
    total_written = 0
    total_removed = 0

    for chunk_idx, chunk in enumerate(
        pd.read_csv(
            input_file,
            chunksize=chunk_size,
            dtype=dtype_map,
            low_memory=False
        ),
        start=1
    ):
        rows_before = len(chunk)
        total_read += rows_before

        row_hashes = pd.util.hash_pandas_object(chunk, index=False)

        # Keep first occurrence inside current chunk
        not_duplicate_in_chunk = ~row_hashes.duplicated(keep="first")

        # Keep only rows not already seen in previous chunks
        not_seen_before = ~row_hashes.isin(seen_hashes)

        keep_mask = not_duplicate_in_chunk & not_seen_before
        chunk_kept = chunk.loc[keep_mask].copy()

        # Store all unique hashes from this chunk
        seen_hashes.update(row_hashes[not_duplicate_in_chunk].tolist())

        rows_after = len(chunk_kept)
        removed_here = rows_before - rows_after

        total_written += rows_after
        total_removed += removed_here

        chunk_kept.to_csv(
            output_file,
            mode="w" if first_write else "a",
            header=first_write,
            index=False,
            encoding="utf-8"
        )

        first_write = False

        print(
            f"Chunk {chunk_idx} -> "
            f"read: {rows_before:,} | kept: {rows_after:,} | "
            f"removed: {removed_here:,} | total kept: {total_written:,}"
        )

    print("\n✅ Done")
    print(f"Saved file: {output_file}")
    print(f"Total rows read: {total_read:,}")
    print(f"Total duplicates removed: {total_removed:,}")
    print(f"Total rows written: {total_written:,}")


remove_duplicates_fast(INPUT_FILE, OUTPUT_FILE, CHUNK_SIZE)

Chunk 1 -> read: 500,000 | kept: 477,529 | removed: 22,471 | total kept: 477,529
Chunk 2 -> read: 500,000 | kept: 484,432 | removed: 15,568 | total kept: 961,961
Chunk 3 -> read: 500,000 | kept: 480,034 | removed: 19,966 | total kept: 1,441,995
Chunk 4 -> read: 500,000 | kept: 473,176 | removed: 26,824 | total kept: 1,915,171
Chunk 5 -> read: 500,000 | kept: 469,155 | removed: 30,845 | total kept: 2,384,326
Chunk 6 -> read: 500,000 | kept: 468,669 | removed: 31,331 | total kept: 2,852,995
Chunk 7 -> read: 500,000 | kept: 470,598 | removed: 29,402 | total kept: 3,323,593
Chunk 8 -> read: 500,000 | kept: 459,099 | removed: 40,901 | total kept: 3,782,692
Chunk 9 -> read: 500,000 | kept: 458,530 | removed: 41,470 | total kept: 4,241,222
Chunk 10 -> read: 500,000 | kept: 462,012 | removed: 37,988 | total kept: 4,703,234
Chunk 11 -> read: 500,000 | kept: 469,216 | removed: 30,784 | total kept: 5,172,450
Chunk 12 -> read: 500,000 | kept: 457,631 | removed: 42,369 | total kept: 5,630,081
Chunk

### **SHAPE & HEAD - AFTER Dealing with Duplicates**

In [30]:
import pandas as pd

FILE = "../data/processed/DUPLICATES_ValeursFoncieres.csv"
CHUNK_SIZE = 500_000

dtype_map = {
    # categorical / identifiers
    "transaction_id": "string",
    "transaction_date": "string",  # convert to datetime after loading
    "transaction_type": "string",
    "street_number": "string",
    "btq_code": "string",
    "street_type": "string",
    "street_id": "string",
    "street_name": "string",
    "postal_code": "string",
    "com_name": "string",
    "dep_code": "string",
    "com_code": "string",
    "section_prefix": "string",
    "section": "string",
    "plot_number": "string",
    "volume_number": "string",
    "property_type_code": "string",  # keep as string (categorical)
    "property_type": "string",
    "land_nature": "string",
    "land_nature_special": "string",
    "surface_type": "string",

    # lot identifiers
    "lot_1": "string",
    "lot_2": "string",
    "lot_3": "string",
    "lot_4": "string",
    "lot_5": "string",

    # numeric (use nullable Float64 for consistency)
    "property_value": "Float64",
    "building_real_surface": "Float64",
    "land_surface": "Float64",
    "lot_1_surface": "Float64",
    "lot_2_surface": "Float64",
    "lot_3_surface": "Float64",
    "lot_4_surface": "Float64",
    "lot_5_surface": "Float64",

    # integer (nullable)
    "lots_count": "Int64",
    "main_rooms_count": "Int64",
}

# ---------------------------
# Get full dataset shape
# ---------------------------
total_rows = 0
columns = None

for chunk in pd.read_csv(
    FILE,
    chunksize=CHUNK_SIZE,
    dtype=dtype_map,
    low_memory=False
):
    total_rows += len(chunk)
    if columns is None:
        columns = chunk.columns

print("Shape:", total_rows, "rows and", len(columns), "columns")

# ---------------------------
# Display head with correct dtypes
# ---------------------------
df_head = pd.read_csv(
    FILE,
    nrows=5,
    dtype=dtype_map,
    low_memory=False
)

display(df_head)

Shape: 18519810 rows and 36 columns


,transaction_id,transaction_date,transaction_type,property_value,street_number,btq_code,street_type,street_id,street_name,postal_code,com_name,dep_code,com_code,section_prefix,section,plot_number,volume_number,lot_1,lot_1_surface,lot_2,lot_2_surface,lot_3,lot_3_surface,lot_4,lot_4_surface,lot_5,lot_5_surface,lots_count,property_type_code,property_type,building_real_surface,main_rooms_count,land_nature,land_nature_special,land_surface,surface_type
0,000001,2020-07-01,Sale,31234.16,Unknown,Unknown,Unknown,B064,SAINT JULIEN,1560,SAINT-JULIEN-SUR-REYSSOUZE,01,367,Unknown,A,8,Unknown,Unknown,0.0,Unknown,0.0,Unknown,0.0,Unknown,0.0,Unknown,0.0,0,Unknown,Unknown,0.0,0,AB,Unknown,1192.0,land
1,000001,2020-07-01,Sale,278000.0,Unknown,Unknown,Unknown,B188,A LA PEROUSE,1250,CORVEISSIAT,01,125,Unknown,C,509,Unknown,Unknown,0.0,Unknown,0.0,Unknown,0.0,Unknown,0.0,Unknown,0.0,0,Unknown,Unknown,0.0,0,BS,Unknown,10092.0,land
2,000001,2020-07-01,Sale,278000.0,Unknown,Unknown,Unknown,B188,A LA PEROUSE,1250,CORVEISSIAT,01,125,Unknown,C,510,Unknown,Unknown,0.0,Unknown,0.0,Unknown,0.0,Unknown,0.0,Unknown,0.0,0,Unknown,Unknown,0.0,0,L,Unknown,4570.0,land
3,000001,2020-07-01,Sale,278000.0,Unknown,Unknown,Unknown,B079,AUX COMMUNS,1250,CORVEISSIAT,01,125,Unknown,ZL,96,Unknown,Unknown,0.0,Unknown,0.0,Unknown,0.0,Unknown,0.0,Unknown,0.0,0,Unknown,Unknown,0.0,0,BS,Unknown,5750.0,land
4,000001,2020-07-01,Sale,278000.0,Unknown,Unknown,Unknown,B033,EN COMBARNAUD,1250,SIMANDRE-SUR-SURAN,01,408,Unknown,A,14,Unknown,Unknown,0.0,Unknown,0.0,Unknown,0.0,Unknown,0.0,Unknown,0.0,0,Unknown,Unknown,0.0,0,BT,Unknown,648170.0,land


### **UNIQUE VALUES & NULLS**

In [38]:
from collections import defaultdict
import pandas as pd

FILE = "../data/processed/DUPLICATES_ValeursFoncieres.csv"

unique_values = defaultdict(set)
null_counts = defaultdict(int)
total_rows = 0

for chunk in pd.read_csv(
    FILE,
    chunksize=500_000,
    dtype=str,          # ✅ avoids mixed types warning
    low_memory=False
):
    total_rows += len(chunk)

    for col in chunk.columns:
        # ✅ Count nulls (DO NOT drop them)
        null_counts[col] += chunk[col].isna().sum()

        # ✅ Collect only non-null values (nulls are still preserved in dataset)
        unique_values[col].update(
            chunk.loc[chunk[col].notna(), col].unique()
        )

# --------------------------------------------------
# Display results
# --------------------------------------------------
print(f"\nTotal rows processed: {total_rows:,}")
print(100 * "-")

for col in unique_values:
    values = list(unique_values[col])
    null_pct = (null_counts[col] / total_rows) * 100

    print(f"➡️ {col}:")
    print(f"Unique values count: {len(values)}")
    print(f"Null count: {null_counts[col]} ({null_pct:.2f}%)")
    print(f"Sample values: {values[:120]}")
    print(100 * "-")


Total rows processed: 18,519,810
----------------------------------------------------------------------------------------------------
➡️ transaction_id:
Unique values count: 311
Null count: 0 (0.00%)
Sample values: ['000189', '000145', '000261', '000115', '000232', '000183', '000141', '000085', '000164', '000126', '000308', '000283', '000043', '000122', '000153', '000055', '000031', '000223', '000143', '000229', '000167', '000022', '000074', '000051', '000200', '000219', '000150', '000015', '000111', '000291', '000007', '000274', '000235', '000068', '000277', '000053', '000166', '000147', '000216', '000154', '000014', '000222', '000311', '000246', '000046', '000021', '000255', '000079', '000162', '000227', '000276', '000295', '000244', '000243', '000161', '000056', '000149', '000280', '000250', '000034', '000116', '000253', '000170', '000080', '000301', '000247', '000176', '000137', '000088', '000086', '000271', '000182', '000281', '000087', '000027', '000019', '000032', '000071', '00

### Resetting the Index

When removing duplicates, the DataFrame index may have gaps due to removed rows. To reset the index after removing duplicates, we can use the `reset_index()` method with the `drop=True` parameter.


In [ ]:
df_without_duplicates.reset_index(drop=True, inplace=True)
df_without_duplicates.tail()

## Setting the index

To set an index in pandas, you can use the `set_index()` method of the DataFrame. This method allows you to specify which column you want to use as the index for the DataFrame.

In [ ]:
df.set_index('PassengerId',inplace=True)
df.head()

<hr>

## 6️⃣ OUTLIERS


<style>
h1 {
    text-align: center;
    color: black;
    font-weight: bold;
}
</style>

<style>
h2 {
    text-align: left;
    color: black;
    font-weight: bold;
}
</style>

<style>
h3 {
    text-align: left;
    color: black;
    font-weight: bold;
}
</style>

<style>
h4 {
    text-align: left;
    color: black;
    font-weight: bold;
}
</style>
<hr>

 An outlier is an observation that lies an abnormal distance from other values in a random sample from a population

**Outlier Analysis**

Purpose →  Identify extreme values that may negatively impact model performance and decide how to properly handle them to improve data quality and model reliability. 

**Outliers can:**
- Skew statistical measures (mean, standard deviation)
- Distort model training (especially for linear models)
- Reduce predictive performance
- Sometimes represent important real-world insights

**We should:**
- keep outliers for Analysis. In this dataset ValeurFoncieres, outliers are insight and not noise.
- deal with outliers for Machine Learning. 

**We expect strong outliers in:**
- `property_value` → luxury sales
- `land_surface` → farms / huge land
- `building_real_surface` → industrial buildings

**We use `IQR` to detect outliers & treat extreme values.**
- top (Q3 + 1.5 x IQR)
- bottom (Q1 - 1.5 x IQR)

**We use BOX PLOTS, SCATTER PLOTS to visualise and detect outliers**


**Because the data:**
- is highly skewed
- has many zeros
- has wide value range

    👉 ALWAYS use: `np.log1p()` otherwise plots are useless.

In [ ]:
import pandas as pd
import numpy as np
import gc

INPUT_FILE = "../data/processed/DUPLICATES_ValeursFoncieres.csv"
OUTPUT_FILE = "../data/processed/OUTLIERS_ValeursFoncieres.csv"

CHUNKSIZE = 500_000
DATE_COL = "transaction_date"
DATE_FORMAT = "%d/%m/%Y"

dtype_map = {
    "transaction_id": "string",
    "transaction_date": "string",
    "transaction_type": "string",
    "street_number": "string",
    "btq_code": "string",
    "street_type": "string",
    "street_id": "string",
    "street_name": "string",
    "postal_code": "string",
    "com_name": "string",
    "dep_code": "string",
    "com_code": "string",
    "section_prefix": "string",
    "section": "string",
    "plot_number": "string",
    "volume_number": "string",
    "property_type_code": "string",
    "property_type": "string",
    "land_nature": "string",
    "land_nature_special": "string",
    "surface_type": "string",
    "lot_1": "string",
    "lot_2": "string",
    "lot_3": "string",
    "lot_4": "string",
    "lot_5": "string",
    "property_value": "Float64",
    "building_real_surface": "Float64",
    "land_surface": "Float64",
    "lot_1_surface": "Float64",
    "lot_2_surface": "Float64",
    "lot_3_surface": "Float64",
    "lot_4_surface": "Float64",
    "lot_5_surface": "Float64",
    "lots_count": "Int64",
    "main_rooms_count": "Int64",
}

GROUP_COLS = ["property_type", "dep_code", "year"]

ZERO_AS_MISSING = [
    "building_real_surface",
    "land_surface",
    "main_rooms_count",
    "lots_count",
    "lot_1_surface",
    "lot_2_surface",
    "lot_3_surface",
    "lot_4_surface",
    "lot_5_surface",
]

NUMERIC_WORK_COLS = [
    "property_value",
    "building_real_surface",
    "land_surface",
    "main_rooms_count",
    "lots_count",
    "lot_1_surface",
    "lot_2_surface",
    "lot_3_surface",
    "lot_4_surface",
    "lot_5_surface",
]

THRESHOLD_COLS = [
    "property_value",
    "price_per_m2_building",
    "land_price_per_m2",
    "building_real_surface_clean",
    "land_surface_clean",
    "main_rooms_count_clean",
    "lots_count_clean",
]


def read_chunks():
    return pd.read_csv(
        INPUT_FILE,
        chunksize=CHUNKSIZE,
        dtype=dtype_map,
        low_memory=False,
        keep_default_na=False,
        na_values=[],
    )


def prepare_chunk(chunk: pd.DataFrame) -> pd.DataFrame:
    chunk["year"] = pd.to_datetime(
        chunk[DATE_COL],
        format=DATE_FORMAT,
        errors="coerce"
    ).dt.year.astype("float64")

    for col in NUMERIC_WORK_COLS:
        if col in chunk.columns:
            chunk[col] = pd.to_numeric(chunk[col], errors="coerce").astype("float64")

    for col in ZERO_AS_MISSING:
        if col in chunk.columns:
            chunk[f"{col}_clean"] = chunk[col].mask(chunk[col] == 0, np.nan)
            chunk[f"flag_zero_{col}"] = chunk[col].eq(0).fillna(False)

    chunk["price_per_m2_building"] = (
        chunk["property_value"] / chunk["building_real_surface_clean"]
    )

    chunk["land_price_per_m2"] = (
        chunk["property_value"] / chunk["land_surface_clean"]
    )

    chunk["building_to_land_ratio"] = (
        chunk["building_real_surface_clean"] / chunk["land_surface_clean"]
    )

    chunk["surface_per_room"] = (
        chunk["building_real_surface_clean"] / chunk["main_rooms_count_clean"]
    )

    chunk["log_property_value"] = np.log1p(chunk["property_value"])

    chunk["flag_multi_lot"] = chunk["lots_count"].ge(2).fillna(False)
    chunk["flag_high_lot_count"] = chunk["lots_count"].ge(5).fillna(False)

    return chunk


print("Pass 1/2: computing thresholds from the full dataset...")

threshold_parts = []

for i, chunk in enumerate(read_chunks(), start=1):
    chunk = prepare_chunk(chunk)

    threshold_parts.append(
        chunk[GROUP_COLS + THRESHOLD_COLS].copy()
    )

    print(f"  processed chunk {i}")

    del chunk
    gc.collect()

threshold_base = pd.concat(threshold_parts, ignore_index=True)

thresholds = (
    threshold_base
    .groupby(GROUP_COLS, dropna=False)
    .agg(
        group_n=("property_value", "size"),

        value_p01=("property_value", lambda x: x.quantile(0.01)),
        value_p99=("property_value", lambda x: x.quantile(0.99)),

        ppm2_p01=("price_per_m2_building", lambda x: x.quantile(0.01)),
        ppm2_p99=("price_per_m2_building", lambda x: x.quantile(0.99)),

        land_ppm2_p01=("land_price_per_m2", lambda x: x.quantile(0.01)),
        land_ppm2_p99=("land_price_per_m2", lambda x: x.quantile(0.99)),

        building_p01=("building_real_surface_clean", lambda x: x.quantile(0.01)),
        building_p99=("building_real_surface_clean", lambda x: x.quantile(0.99)),

        land_p01=("land_surface_clean", lambda x: x.quantile(0.01)),
        land_p99=("land_surface_clean", lambda x: x.quantile(0.99)),

        rooms_p01=("main_rooms_count_clean", lambda x: x.quantile(0.01)),
        rooms_p99=("main_rooms_count_clean", lambda x: x.quantile(0.99)),

        lots_p01=("lots_count_clean", lambda x: x.quantile(0.01)),
        lots_p99=("lots_count_clean", lambda x: x.quantile(0.99)),
    )
    .reset_index()
)

del threshold_base
del threshold_parts
gc.collect()

print("Thresholds computed.")
print("Pass 2/2: removing outliers and writing cleaned dataset...")

first_write = True
total_rows = 0
kept_rows = 0
removed_rows = 0

for i, chunk in enumerate(read_chunks(), start=1):
    chunk = prepare_chunk(chunk)

    chunk = chunk.merge(
        thresholds,
        on=GROUP_COLS,
        how="left"
    )

    chunk["flag_property_value_outlier"] = (
        (chunk["property_value"] < chunk["value_p01"]) |
        (chunk["property_value"] > chunk["value_p99"])
    ).fillna(False)

    chunk["flag_price_m2_outlier"] = (
        (chunk["price_per_m2_building"] < chunk["ppm2_p01"]) |
        (chunk["price_per_m2_building"] > chunk["ppm2_p99"])
    ).fillna(False)

    chunk["flag_land_price_m2_outlier"] = (
        (chunk["land_price_per_m2"] < chunk["land_ppm2_p01"]) |
        (chunk["land_price_per_m2"] > chunk["land_ppm2_p99"])
    ).fillna(False)

    chunk["flag_building_surface_outlier"] = (
        (chunk["building_real_surface_clean"] < chunk["building_p01"]) |
        (chunk["building_real_surface_clean"] > chunk["building_p99"])
    ).fillna(False)

    chunk["flag_land_surface_outlier"] = (
        (chunk["land_surface_clean"] < chunk["land_p01"]) |
        (chunk["land_surface_clean"] > chunk["land_p99"])
    ).fillna(False)

    chunk["flag_rooms_outlier"] = (
        (chunk["main_rooms_count_clean"] < chunk["rooms_p01"]) |
        (chunk["main_rooms_count_clean"] > chunk["rooms_p99"])
    ).fillna(False)

    chunk["flag_lots_count_outlier"] = (
        (chunk["lots_count_clean"] < chunk["lots_p01"]) |
        (chunk["lots_count_clean"] > chunk["lots_p99"])
    ).fillna(False)

    chunk["flag_extreme_value"] = chunk["property_value"].gt(50_000_000).fillna(False)
    chunk["flag_extreme_building_surface"] = chunk["building_real_surface_clean"].gt(5_000).fillna(False)
    chunk["flag_extreme_land_surface"] = chunk["land_surface_clean"].gt(100_000).fillna(False)
    chunk["flag_extreme_rooms"] = chunk["main_rooms_count_clean"].gt(50).fillna(False)
    chunk["flag_extreme_lots"] = chunk["lots_count_clean"].gt(20).fillna(False)

    chunk["flag_tiny_room_surface"] = chunk["surface_per_room"].lt(5).fillna(False)
    chunk["flag_huge_room_surface"] = chunk["surface_per_room"].gt(150).fillna(False)

    outlier_flags = [
        "flag_property_value_outlier",
        "flag_price_m2_outlier",
        "flag_land_price_m2_outlier",
        "flag_building_surface_outlier",
        "flag_land_surface_outlier",
        "flag_rooms_outlier",
        "flag_lots_count_outlier",
        "flag_extreme_value",
        "flag_extreme_building_surface",
        "flag_extreme_land_surface",
        "flag_extreme_rooms",
        "flag_extreme_lots",
        "flag_tiny_room_surface",
        "flag_huge_room_surface",
    ]

    chunk["is_outlier"] = chunk[outlier_flags].fillna(False).any(axis=1)

    chunk_clean = chunk.loc[~chunk["is_outlier"]].copy()

    total_rows += len(chunk)
    kept_rows += len(chunk_clean)
    removed_rows += len(chunk) - len(chunk_clean)

    chunk_clean.to_csv(
        OUTPUT_FILE,
        mode="w" if first_write else "a",
        index=False,
        header=first_write
    )

    print(f"  chunk {i}: kept {len(chunk_clean):,} / {len(chunk):,} rows")

    first_write = False

    del chunk
    del chunk_clean
    gc.collect()

print("Done.")
print(f"Input rows processed: {total_rows:,}")
print(f"Rows kept: {kept_rows:,}")
print(f"Rows removed as outliers: {removed_rows:,}")
print(f"Saved cleaned dataset to: {OUTPUT_FILE}")

Pass 1/2: computing thresholds from the full dataset...
  processed chunk 1
  processed chunk 2
  processed chunk 3
  processed chunk 4
  processed chunk 5
  processed chunk 6
  processed chunk 7
  processed chunk 8
  processed chunk 9
  processed chunk 10
  processed chunk 11
  processed chunk 12
  processed chunk 13
  processed chunk 14
  processed chunk 15
  processed chunk 16
  processed chunk 17
  processed chunk 18
  processed chunk 19
  processed chunk 20
  processed chunk 21
  processed chunk 22
  processed chunk 23
  processed chunk 24
  processed chunk 25
  processed chunk 26
  processed chunk 27
  processed chunk 28
  processed chunk 29
  processed chunk 30
  processed chunk 31
  processed chunk 32
  processed chunk 33
  processed chunk 34
  processed chunk 35
  processed chunk 36
  processed chunk 37
  processed chunk 38
Thresholds computed.
Pass 2/2: removing outliers and writing cleaned dataset...
  chunk 1: kept 476,756 / 500,000 rows
  chunk 2: kept 478,019 / 500,000 ro

---
### **DETECT OUTLIERS (METHODS)**

---
#### **Method 1: IQR (Interquartile Range)**
- Calculate Q1 (25th percentile)
- Calculate Q3 (75th percentile)
- Compute IQR = Q3 − Q1
- Define bounds:
    - Lower bound = Q1 − 1.5 × IQR
    - Upper bound = Q3 + 1.5 × IQR
    - Values outside these bounds are considered **outliers**

Tukey's test for outliers involves using the interquartile range (IQR) to determine if values are outliers. While there isn't a direct "Tukey's test" function in Python's standard libraries, you can easily compute it using the IQR.

Here's how you can perform Tukey's test for outliers:

1. Calculate the first quartile (Q1) and third quartile (Q3) for the data.
2. Compute the interquartile range: \( \text{IQR} = Q3 - Q1 \).
3. Identify outliers:
   - Values less than \( Q1 - 1.5 \times \text{IQR} \)
   - Values greater than \( Q3 + 1.5 \times \text{IQR} \)

Here's a Python function that implements Tukey's test for outliers:

In [ ]:
def tukeys_test_outliers(data):
    Q1 = data.quantile(0.25)
    Q3 = data.quantile(0.75)
    IQR = Q3 - Q1
    
    # Define bounds for the outliers
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    # Identify the outliers
    outliers = data[(data < lower_bound) | (data > upper_bound)]
    
    return outliers

This function returns the values in the data that are considered outliers according to Tukey's method.

In [ ]:
# Example usage:
data_series = df['ColumnName']  # replace 'ColumnName' with your specific column
outliers = tukeys_test_outliers(data_series)
print(outliers)

If you want to **modify the outlier values**, you can replace them with a specific value, such as the median, or the bounds defined by Tukey's method.

If you wish to **delete the outliers**, you can simply filter them out.

Implement these three approaches modifying the function above.

---
#### **Method 2: Z-score**
- Calculate mean and standard deviation
- Compute Z-score for each value
- Values with |Z| > 3 are typically considered **outliers**

---
#### **Before removing anything, check:**
- Are they data entry errors?
- Are they rare but valid observations?
- Do they represent a specific subgroup?
- How many outliers are there (percentage of the data)?

---
#### **Then Decide whether to:**

**a. Keep outliers:**
- If they are valid and meaningful
- If they represent real-world variability

**b. Transform outliers:**
- Apply log transformation
- Apply scaling (RobustScaler)
- Apply winsorization (cap extreme values)

**c. Remove outliers:**
- If they are clear errors
- If they severely distort the model
- If they represent noise rather than signal

---
### **PLOTS**

#### **BOX PLOTS: Side by Side**

Box plots are a great way to visualize the distribution of a continuous variable across different categories. They show the median, quartiles, and potential outliers, providing a quick snapshot of the data's spread and central tendency.

Let's visualize the distribution of `property_value` for each column category using side-by-side box plots. This will help us understand the spread, median, and potential outliers for each category.

In [ ]:
sns.boxplot(data=df, x='MSZoning', y='SalePrice', palette="coolwarm")

- There are noticeable outliers in the `A` and `B` categories, with some houses selling at values much higher than the typical range for those categories.

### **SCATTER PLOTS**

**Potential Outliers**: There are a few houses with large surface areas but lower sale values. These points could be potential outliers or could represent houses in areas where value per square meter is lower.

<hr>

## 7️⃣ CLEAN


<style>
h1 {
    text-align: center;
    color: black;
    font-weight: bold;
}
</style>

<style>
h2 {
    text-align: left;
    color: black;
    font-weight: bold;
}
</style>

<style>
h3 {
    text-align: left;
    color: black;
    font-weight: bold;
}
</style>

<style>
h4 {
    text-align: left;
    color: black;
    font-weight: bold;
}
</style>
<hr>

ONE SCRIPT FOR CLEANING DATA & SAVING IT as CLEAN_ValeursFoncieres.csv